# NoiPA Dataset Quality Pipeline

**A multi-agent system for validating and cleaning Italian public-administration CSV datasets.**

This notebook explains the pipeline end-to-end and shows the real production code used to run it. It is **runnable top-to-bottom** on `Data/spesa.csv`: every cell you see below executes the real production code, writes the same artifacts the CLI does, and produces a Markdown narrative report at the end.

The pipeline is organised in two halves:

* **Validation** (read-only) — infers per-column dtypes, measures completeness, detects format inconsistencies, surfaces anomalies, checks cross-column coherence, and finds duplicate records.
* **Cleaning** — derives a deterministic remediation plan from the validation findings, then runs a **generator / critic** loop that synthesises one Python cleaning function per inconsistent column, applies the plan plus the generated cleaners, verifies the result, and writes a narrative report.

All agents are [Pydantic AI](https://ai.pydantic.dev) agents backed by `openai-responses:gpt-5.4-mini`. Every structured output is a Pydantic model: the prompt states *what* to do, the Pydantic schema states *how* the answer must be shaped, and host-side code owns correctness checks and retries.


## 1. Architecture overview

```
                +---------------------+
                |   cli.py / main.py  |
                +----------+----------+
                           |
                           v
            +--------------+-------------+
            |        validation/         |     VALIDATION HALF
            |                            |
            | dtype -> schema ->         |
            | completeness -> consistency|
            | -> anomaly -> cross-column |
            | -> duplicates              |
            +--------------+-------------+
                           |
                           v
            +--------------+-------------+
            |  cleaning/                 |     CLEANING HALF
            |  orchestrator.run_cleaning |
            |                            |
            |  remediation ->            |
            |  generation (gen/critic) ->|
            |  application ->            |
            |  verification ->           |
            |  reporting                 |
            +----------------------------+
```

### The twelve production agents

| # | Agent | Role |
|---|---|---|
| 1 | `dtype_inference_agent` | Infers the cleaned pandas dtype, semantic role, and dominant target pattern per column. |
| 2 | `schema_summary_agent` | Narrates schema issues, naming fixes, and duplicate-semantic groups. |
| 3 | `completeness_analysis_agent` | Turns the completeness profile into a structured completeness report. |
| 4 | `format_consistency_agent` | Slow-path per-column inconsistency detector when schema fast-path rules are insufficient. |
| 5 | `column_cleaner_generator_agent` | Writes one self-contained Python cleaner for one inconsistent column. |
| 6 | `cleaner_repair_critic_agent` | Diagnoses failed cleaners and prescribes the next repair. |
| 7 | `anomaly_summary_agent` | Narrates numeric-outlier and rare-category findings. |
| 8 | `cross_column_summary_agent` | Narrates duplicate-column, semantic-conflict, and period/date-order findings. |
| 9 | `duplicate_summary_agent` | Narrates exact and near-duplicate row groups. |
| 10 | `narrative_report_agent` | Legacy monolithic narrative writer retained in `agents.py` for inspection. |
| 11 | `narrative_frontmatter_agent` | Writes the report title, executive summary, and recommendations. |
| 12 | `narrative_section_agent` | Writes one grounded report section at a time for the chunked reporting pipeline. |

### Artifacts produced

All paths are relative to the dataset's parent directory.

* `Data/.validation_cache/<dataset>.schema_handoff.json`
* `Data/.validation_cache/<dataset>.completeness.json`
* `Data/.validation_cache/<dataset>.consistency.json`
* `Data/.validation_cache/<dataset>.anomaly.json`
* `Data/.validation_cache/<dataset>.cross_column.json`
* `Data/.validation_cache/<dataset>.duplicates.json`
* `Data/.validation_cache/<dataset>.remediation_plan.json`
* `Data/.validation_cache/<dataset>.validation_bundle.json`
* `Data/.cleaning_cache/<dataset>/generated_cleaners/*.py`
* `Data/.cleaning_cache/<dataset>/cleaner_manifest.json`
* `Data/.cleaning_cache/<dataset>/<dataset>.cleaned.csv`
* `Data/.cleaning_cache/<dataset>/<dataset>.final_report.json`
* `Data/.cleaning_cache/<dataset>/<dataset>.narrative_report.md`


### 1.1 Interactive pipeline diagram

The same flow above, rendered with `graphviz` so it can be read at a glance. Nodes are coloured by half of the pipeline: **yellow** = validation stages (read-only, profile + agent summaries), **pink** = deterministic planning, **green** = cleaning (generator / critic / host validator / application), **blue** = verification and reporting.

Requires the `graphviz` Python package and the Graphviz binaries on PATH (install from <https://graphviz.org/download/>).

In [ ]:
# Ensure the Graphviz `dot` binary is findable. If you installed Graphviz to one of the
# standard Windows locations but forgot to tick 'Add to PATH', this prepends it for this session.
import os, shutil
if shutil.which("dot") is None:
    for candidate in [r"C:\Program Files\Graphviz\bin", r"C:\Program Files (x86)\Graphviz\bin"]:
        if os.path.isfile(os.path.join(candidate, "dot.exe")):
            os.environ["PATH"] = candidate + os.pathsep + os.environ.get("PATH", "")
            break

from graphviz import Digraph

g = Digraph(
    "NoiPA pipeline",
    graph_attr={
        "rankdir": "LR",
        "bgcolor": "white",
        "pad": "0.35",
        "nodesep": "0.45",
        "ranksep": "0.75",
        "splines": "spline",
        "fontname": "Helvetica",
    },
)

g.attr(
    "node",
    shape="box",
    style="rounded,filled",
    fontname="Helvetica",
    fontsize="10",
    margin="0.18,0.10",
    penwidth="1.2",
)

g.attr(
    "edge",
    fontname="Helvetica",
    fontsize="9",
    color="#666666",
    arrowsize="0.8",
    penwidth="1.1",
)

# --- Validation half (warm yellow) ---
val_fill = "#fff6cc"
val_border = "#d4a017"
with g.subgraph(name="cluster_validation") as v:
    v.attr(
        label="Validation  ·  read-only profiling",
        style="rounded,filled",
        color=val_border,
        fillcolor="#fffdf4",
        pencolor=val_border,
        fontname="Helvetica-Bold",
        fontsize="13",
        margin="16",
    )
    for node in [
        ("dtype",        "run_dtype_inference"),
        ("schema",       "run_schema_validation"),
        ("completeness", "run_completeness_analysis"),
        ("consistency",  "run_format_consistency_validation"),
        ("anomaly",      "run_anomaly_detection"),
        ("cross",        "run_cross_column_validation"),
        ("dupes",        "run_duplicate_detection"),
        ("bundle",       "build_validation_results"),
    ]:
        v.node(node[0], node[1], fillcolor=val_fill, color=val_border)

# --- Remediation (soft rose) ---
g.node(
    "remediation",
    "run_remediation_planning\n(deterministic, no LLM)",
    fillcolor="#fde2e4",
    color="#d48a98",
)

# --- Cleaning half (soft green) ---
clean_fill = "#dff3e4"
clean_border = "#3f8f5a"
with g.subgraph(name="cluster_cleaning") as c:
    c.attr(
        label="Cleaning  ·  generator / critic / host validator",
        style="rounded,filled",
        color=clean_border,
        fillcolor="#f5fbf6",
        pencolor=clean_border,
        fontname="Helvetica-Bold",
        fontsize="13",
        margin="16",
    )
    for node in [
        ("request",    "build_column_cleaning_request"),
        ("generator",  "column_cleaner_generator_agent"),
        ("validator",  "validate_generated_cleaner_program\n(pure Python, no LLM)"),
        ("critic",     "cleaner_repair_critic_agent"),
        ("apply",      "run_cleaner_application_with_plan"),
    ]:
        c.node(node[0], node[1], fillcolor=clean_fill, color=clean_border)

# --- Verification + reporting (soft blue) ---
rep_fill = "#dcecff"
rep_border = "#4f83cc"
with g.subgraph(name="cluster_report") as r:
    r.attr(
        label="Verification + reporting",
        style="rounded,filled",
        color=rep_border,
        fillcolor="#f7fbff",
        pencolor=rep_border,
        fontname="Helvetica-Bold",
        fontsize="13",
        margin="16",
    )
    r.node("verify",    "run_verify",                fillcolor=rep_fill, color=rep_border)
    r.node("final",     "build_final_report",        fillcolor=rep_fill, color=rep_border)
    r.node("narrative", "generate_narrative_report", fillcolor=rep_fill, color=rep_border)

# --- Edges: the actual dataflow ---
g.edge("dtype", "schema", label="DatasetDtypeInference")

for stage in ["schema", "completeness", "consistency", "anomaly", "cross", "dupes"]:
    g.edge(stage, "bundle")

g.edge("bundle", "remediation", label="OrchestrationStepResult")

g.edge("consistency", "request", label="FormatConsistencyFinding")
g.edge("schema",      "request", label="SchemaColumnEntry")

g.edge("request",   "generator", label="ColumnCleaningRequest")
g.edge("generator", "validator", label="ColumnCleanerProgram")

g.edge(
    "validator", "critic",
    label="validation issues",
    color="#c62828",
    fontcolor="#c62828",
)

g.edge(
    "critic", "generator",
    label="CleanerRepairDiagnosis · retry",
    color="#c62828",
    fontcolor="#c62828",
    style="dashed",
)

g.edge(
    "validator", "apply",
    label="accepted program",
    color="#2e7d32",
    fontcolor="#2e7d32",
)

g.edge("remediation", "apply", label="RemediationPlan")

g.edge("apply",  "verify",    label="cleaned.csv")
g.edge("verify", "final",     label="ConsistencyVerificationReport")
g.edge("bundle", "final",     label="findings", style="dashed", constraint="false")
g.edge("apply",  "final",     label="CleaningReport", style="dashed", constraint="false")
g.edge("final",  "narrative", label="FinalPipelineReport")

g


## 2. Setup

This setup cell loads the API key, patches the notebook event loop so pydantic-ai can run cleanly inside Jupyter, and enables Logfire tracing. The lower-level plumbing stays inside the production modules and is exercised later through the stage functions shown in the notebook.


In [1]:
# Import standard libraries and third-party dependencies
from __future__ import annotations
import asyncio
import pandas as pd
import json
import inspect
import nest_asyncio
import sys
from pathlib import Path
from dotenv import load_dotenv        
from IPython.display import Markdown, display
from agents import setup_logfire
from pydantic import BaseModel, Field, field_validator
from typing import Any, Literal



# Pydantic AI agents call asyncio.run() internally; Jupyter already has running loop, so we patch it to allow re-entry.
nest_asyncio.apply()

# Read OPENAI_API_KEY from .env if available
load_dotenv()

# Configure Logfire observability
setup_logfire()

# Validation agent imports
from agents import (
    anomaly_summary_agent,
    completeness_analysis_agent,
    cross_column_summary_agent,
    dtype_inference_agent,
    duplicate_summary_agent,
    schema_summary_agent,
)

# Cache import for schema handoff objects and validation artifacts
from cache import (
    load_anomaly,
    load_completeness,
    load_consistency,
    load_cross_column,
    load_duplicates,
    load_schema_handoff,
    save_anomaly,
    save_completeness,
    save_consistency,
    save_cross_column,
    save_duplicates,
    save_schema_handoff,
    save_validation_results,
)

# Validation model imports
from models import (
    AnomalyDetectionReport,
    AnomalyFinding,
    CompletenessAnalysisReport,
    ConsistencyValidationReport,
    CrossColumnFinding,
    CrossColumnValidationReport,
    DatasetDtypeInference,
    DuplicateDetectionReport,
    DuplicateRecordGroup,
    FormatConsistencyFinding,
    OrchestrationStepResult,
    SchemaColumnEntry,
    SchemaHandoff,
)

# Schema tools and utilities imports
from tools.schema_tools import (
    SchemaDuplicateGroup,
    build_dataset_profile,
    build_dtype_inference_text,
    is_valid_schema_name,
    naming_rule_reason,
    normalized_schema_name,
    suggest_schema_name,
)

# Common tools and validation imports
from tools.common_tools import (
    attach_profile_text,
    attach_text_document,
    load_dataset_frame,
    run_agent_with_backoff,
)

# Completeness tools
from tools.completeness_tools import (
    build_completeness_profile,
)


# Quality and detection tools imports
from tools.quality_tools import (
    detect_date_order_violations,
    detect_duplicate_like_columns,
    detect_duplicate_semantic_conflicts,
    detect_exact_duplicate_groups,
    detect_near_duplicate_groups,
    detect_numeric_outlier_candidates,
    detect_rare_category_candidates,
    detect_year_month_period_mismatches,
    infer_duplicate_key_columns,
)

# Validation helper imports
from validation._summary import summarize_validation_report
from validation.anomaly import _duplicate_semantic_suppressed_columns
from validation.consistency import _run_column_format_checks_async, run_column_format_check
from validation.schema import _normalize_dtype_inference_choice, build_schema_issues

# Validation pipeline imports
from validation import (
    run_anomaly_detection,
    run_completeness_analysis,
    run_cross_column_validation,
    run_dtype_inference,
    run_duplicate_detection,
    run_format_consistency_validation,
    run_schema_validation,
)

KeyboardInterrupt: 

## 3. Dataset

We default to `Data/spesa.csv`, a small NoiPA spending dataset (~20k rows). To swap to the larger `Data/attivazioniCessazioni.csv`, change the path below.


In [ ]:
DATASET_PATH = Path("Data/spesa.csv").resolve()
assert DATASET_PATH.exists(), f"dataset not found: {DATASET_PATH}"
print(DATASET_PATH)

In [ ]:
from tools import load_dataset_frame  # thin pd.read_csv wrapper used by every stage
raw_df = load_dataset_frame(DATASET_PATH)
print(f"{len(raw_df):,} rows x {len(raw_df.columns)} columns")
raw_df.head()

## 4. Data contracts (Pydantic models)

This section brings the model layer from `models.py` directly into the notebook. The classes are grouped by responsibility so the reader can understand the contract layer in stages: schema inference, validation findings, cleaning and repair, and final orchestration/reporting.

Taken together, the grouped cells below cover the full project-specific model layer used by the pipeline. The only important external type referenced from outside `models.py` is `SchemaDuplicateGroup`, which comes from `tools` because it is produced by deterministic schema profiling helpers rather than by the model module itself.


### 4.0 Reading map for the model layer

The grouped cells below follow the same flow as the pipeline:

1. Shared modeling primitives: controlled vocabularies and type literals.
2. Schema and dtype inference models: what the pipeline believes each column should be.
3. Validation and verification models: the evidence produced by the validation half and by post-cleaning verification.
4. Cleaning and repair models: the generator input/output contracts and execution logs.
5. Planning, reporting, and orchestration models: how stage-level outputs are merged into final pipeline artifacts.

This organization is more useful didactically than listing models in raw file order, because it mirrors how the pipeline actually uses them.


### 4.1 Shared modeling primitives

Role: this cell shows the shared imports and type literals used throughout the model layer. These literals matter because they enforce a controlled vocabulary for dtypes, semantic roles, remediation actions, and cleaner-validation failures.


In [ ]:
from __future__ import annotations

from typing import Any, Literal
from pydantic import BaseModel, Field, field_validator
from tools import SchemaDuplicateGroup

# Shared type vocabularies used across the pipeline.
VALID_PANDAS_DTYPE = Literal[
    "Int64",
    "Float64",
    "datetime64[ns]",
    "string",
    "boolean",
    "object",
]

NUMERIC_ROLE = Literal[
    "measure",    # real quantity used arithmetically
    "code",       # numeric identifier that should not be treated as a quantity
    "indicator",  # bounded numeric status / ordinal flag
]

STRING_ROLE = Literal[
    "identifier",
    "categorical",
    "name",
    "free_text",
]

REMEDIATION_ACTION_TYPE = Literal[
    "rename_column",
    "replace_placeholders_with_null",
    "generate_cleaner",
    "drop_exact_duplicate_column",
    "cast_dtype",
    "manual_review",
    "report_only",
    "drop_rows_candidate",
]
REMEDIATION_OBJECT_TYPE = Literal["column", "column_pair", "row_group", "dataset"]
REMEDIATION_CONFIDENCE = Literal["low", "medium", "high"]
REMEDIATION_RISK_LEVEL = Literal["low", "medium", "high"]
REMEDIATION_STATUS = Literal["planned", "applied", "proposed_not_applied", "failed", "not_needed"]

VALIDATION_FAILURE_CATEGORY = Literal[
    "program_mismatch",
    "verification_report_failed",
    "non_self_contained_function",
    "runtime_exception",
    "shadowed_specific_branch",
    "dominant_value_modified",
    "outlier_unchanged",
    "outlier_returned_none",
    "unrecoverable_outlier_not_nulled",
    "wrong_output_shape",
    "not_parseable_as_target_dtype",
    "not_matching_target_pattern",
]


### 4.2 Schema and dtype inference models

Role: these models describe the first layer of understanding. They capture what each column appears to be, which naming issues exist, and what cleaned dtype and dominant pattern the pipeline should aim for downstream.


In [ ]:
# --- 4.2 Schema and dtype inference models ---
# Models defined below:
# - SchemaIssue, SchemaColumnEntry        -> atomic schema-level findings
# - SchemaHandoff                         -> compact object passed to later stages
# - ColumnDtypeInference, DatasetDtypeInference -> dtype agent output contract
# - SchemaSummaryOutput                   -> narrator model for the schema summary

class SchemaIssue(BaseModel):
    column_name: str
    issue_type: str = Field(
        description="Use values such as naming_standard, reserved_word_risk, inferred_type_mismatch, or duplicate_column_semantics."
    )
    severity: str = Field(description="Use low, medium, or high.")
    evidence: str
    fix_confidence: str = Field(description="Use high, medium, or low.")
    suggested_fix: str
    suggested_strategy: str = Field(
        description="If fix_confidence is not high, provide a cautious remediation strategy instead of a precise correction."
    )

class SchemaColumnEntry(BaseModel):
    """All facts about one column: dtype inference, statistics, and naming check — merged into one place."""
    name: str
    pandas_dtype: str
    numeric_role: NUMERIC_ROLE | None = None
    string_role: STRING_ROLE | None = None
    detected_pattern: str | None = None
    rationale: str
    non_null_rows: int = Field(ge=0)
    distinct_non_null_values: int = Field(ge=0)
    numeric_parse_pct: float = Field(ge=0, le=100)
    datetime_parse_pct: float = Field(ge=0, le=100)
    empty_like_pct: float = Field(ge=0, le=100)
    sample_values: list[str] = Field(default_factory=list)
    naming_valid: bool
    rename_suggestion: str | None = None
    naming_reason: str | None = None

class SchemaHandoff(BaseModel):
    """Complete schema analysis result. Column-centric: all dtype, statistical, and naming
    facts per column are merged. Issues and duplicate groups are kept at the top level
    for easy scanning by downstream fixing agents."""
    dataset_name: str
    total_rows: int = Field(ge=0)
    total_columns: int = Field(ge=0)
    columns: list[SchemaColumnEntry] = Field(default_factory=list)
    issues: list[SchemaIssue] = Field(default_factory=list)
    duplicate_groups: list[SchemaDuplicateGroup] = Field(default_factory=list)
    summary: str = ""

class SchemaSummaryOutput(BaseModel):
    summary: str

class ColumnDtypeInference(BaseModel):
    column_name: str
    pandas_dtype: VALID_PANDAS_DTYPE
    numeric_role: NUMERIC_ROLE | None = Field(
        default=None,
        description="Only set when pandas_dtype is Int64 or Float64.",
    )
    string_role: STRING_ROLE | None = Field(
        default=None,
        description="Only set when pandas_dtype is string.",
    )
    detected_pattern: str | None = Field(
        default=None,
        description=(
            "Describe the dominant value format when a clear pattern is present. "
            "Examples: 'YYYY-MM', 'DD/MM/YYYY', 'Italian decimal comma (1.234,56)', "
            "'6-digit numeric code', 'ISO 3166-1 alpha-2 country code'. "
            "Leave null when no consistent pattern is detectable."
            "Pick the most common pattern only, no mixed results."
        ),
    )
    rationale: str

class DatasetDtypeInference(BaseModel):
    columns: list[ColumnDtypeInference]


### 4.3 Validation findings and verification models

Role: these models hold the main evidence discovered by the validation half: missingness, format inconsistencies, anomalies, cross-column conflicts, duplicates, and the before/after verification diffs produced after cleaning.


In [ ]:
# --- 4.3 Validation findings and verification models ---
# Models defined below:
# - CompletenessColumnFinding, FormatConsistencyFinding, AnomalyFinding,
#   CrossColumnFinding, DuplicateRecordGroup    -> atomic findings, later aggregated into reports
# - ColumnConsistencyReport                     -> per-column slow-path agent output
# - ConsistencyVerificationReport               -> post-cleaning re-run diff
# - *SummaryOutput classes                      -> narrator contracts for stage summaries

class CompletenessColumnFinding(BaseModel):
    column_name: str
    completeness_pct: float = Field(ge=0, le=100)
    missing_like_count: int = Field(ge=0)
    missing_like_examples: list[str] = Field(default_factory=list)
    sparse_candidate: bool = False
    recommended_action: str

class CompletenessAnalysisReport(BaseModel):
    dataset_name: str
    total_rows: int = Field(ge=0)
    total_columns: int = Field(ge=0)
    overall_completeness_pct: float = Field(ge=0, le=100)
    columns_with_missing_values: list[str] = Field(default_factory=list)
    sparse_columns: list[str] = Field(default_factory=list)
    placeholder_values_detected: list[str] = Field(default_factory=list)
    per_column: list[CompletenessColumnFinding] = Field(default_factory=list)
    summary: str

class FormatConsistencyFinding(BaseModel):
    column_name: str
    expected_pattern: str
    inconsistent_rows: int = Field(ge=0)
    example_inconsistent_values: list[str] = Field(default_factory=list)
    evidence: str
    suggested_strategy: str

class ConsistencyValidationReport(BaseModel):
    dataset_name: str
    total_rows: int = Field(ge=0)
    format_consistency_findings: list[FormatConsistencyFinding] = Field(default_factory=list)
    summary: str

class ColumnConsistencyReport(BaseModel):
    finding: FormatConsistencyFinding | None = None
    summary: str

class FindingDiff(BaseModel):
    column_name: str
    status: Literal["resolved", "improved", "unchanged", "regressed", "new"]
    before_inconsistent_rows: int = Field(ge=0)
    after_inconsistent_rows: int = Field(ge=0)
    reduction_pct: float = Field(ge=-100)
    remaining_examples: list[str] = Field(default_factory=list)
    renamed_to: str | None = None

class ConsistencyVerificationReport(BaseModel):
    dataset_name: str
    original_finding_count: int = Field(ge=0)
    remaining_finding_count: int = Field(ge=0)
    diffs: list[FindingDiff] = Field(default_factory=list)
    summary: str

class AnomalyFinding(BaseModel):
    column_name: str
    anomaly_type: Literal["numeric_outlier", "rare_category"]
    severity: Literal["low", "medium", "high"]
    affected_rows: int = Field(ge=0)
    example_values: list[str] = Field(default_factory=list)
    evidence: str
    suggested_action: str

class AnomalyDetectionReport(BaseModel):
    dataset_name: str
    total_rows: int = Field(ge=0)
    total_columns: int = Field(ge=0)
    findings: list[AnomalyFinding] = Field(default_factory=list)
    summary: str = ""

class AnomalySummaryOutput(BaseModel):
    summary: str

class CrossColumnFinding(BaseModel):
    columns: list[str] = Field(default_factory=list)
    check_type: Literal[
        "duplicate_semantic_conflict",
        "exact_duplicate_columns",
        "near_duplicate_columns",
        "year_month_period_mismatch",
        "date_order_violation",
    ]
    severity: Literal["medium", "high"]
    affected_rows: int = Field(ge=0)
    example_row_indices: list[int] = Field(default_factory=list)
    similarity_pct: float | None = Field(default=None, ge=0, le=100)
    evidence: str
    suggested_action: str

class CrossColumnValidationReport(BaseModel):
    dataset_name: str
    total_rows: int = Field(ge=0)
    findings: list[CrossColumnFinding] = Field(default_factory=list)
    summary: str = ""

class CrossColumnSummaryOutput(BaseModel):
    summary: str

class DuplicateRecordGroup(BaseModel):
    duplicate_type: Literal["exact_row", "near_duplicate"]
    row_indices: list[int] = Field(default_factory=list)
    key_columns: list[str] = Field(default_factory=list)
    evidence: str
    suggested_action: str

class DuplicateDetectionReport(BaseModel):
    dataset_name: str
    total_rows: int = Field(ge=0)
    groups: list[DuplicateRecordGroup] = Field(default_factory=list)
    summary: str = ""

class DuplicateSummaryOutput(BaseModel):
    summary: str


### 4.4 Cleaning, execution, and repair models

Role: these models define the contract of the cleaning half. They specify what one cleaner should receive, what code the generator must return, how host validation reports failures, and how execution results are stored once the cleaner is applied to the dataframe.


In [ ]:
# --- 4.4 Cleaning, execution, and repair models ---
# Models defined below:
# - ColumnCleaningRequest                        -> prompt contract for one dirty column
# - ColumnCleanerProgram                         -> structured generator output (Python code + metadata)
# - CleanerValidationIssue, CleanerRepairContext,
#   CleanerRepairExample, CleanerRepairDiagnosis -> the generator/critic loop, made inspectable
# - CellUpdate, ColumnCleanerExecutionReport     -> what actually changed during application
# - GeneratedCleanerArtifact, CleaningReport     -> accepted cleaners, summarized at dataset level

class ColumnCleaningRequest(BaseModel):
    dataset_name: str
    column_name: str
    expected_pattern: str
    semantic_hint: str
    target_dtype: str | None = None
    target_role: str | None = None
    dominant_shape: str | None = None
    dominant_example_values: list[str] = Field(default_factory=list)
    example_inconsistent_values: list[str] = Field(default_factory=list)
    suggested_strategy: str

class ExampleTransformation(BaseModel):
    original_value: str
    cleaned_value: str | None = None
    rationale: str

    @field_validator("original_value", "cleaned_value", mode="before")
    @classmethod
    def coerce_to_str(cls, v):
        if v is None:
            return None
        return str(v)

class ColumnCleanerProgram(BaseModel):
    column_name: str
    function_name: str
    python_code: str = Field(
        description=(
            "Pure Python source code containing exactly one function definition named function_name. "
            "Must be importable as-is: no test code, no print statements, no variable assignments outside the function, no JSON. "
            "All imports and helper constants must be inside the function body."
        )
    )
    example_transformations: list[ExampleTransformation] = Field(default_factory=list)
    verification_summary: str = ""
    residual_risks: list[str] = Field(default_factory=list)

class CleanerValidationIssue(BaseModel):
    category: VALIDATION_FAILURE_CATEGORY
    severity: Literal["high", "medium"]
    message: str
    input_value: str | None = None
    actual_output: str | None = None
    expected_behavior: str

class CleanerRepairContext(BaseModel):
    request: ColumnCleaningRequest
    previous_program: ColumnCleanerProgram
    validation_issues: list[CleanerValidationIssue] = Field(default_factory=list)

class CleanerRepairExample(BaseModel):
    input_value: str
    actual_output: str | None = None
    expected_output: str | None = None
    fix_note: str

class CleanerRepairDiagnosis(BaseModel):
    should_retry: bool = True
    primary_category: VALIDATION_FAILURE_CATEGORY
    root_cause: str
    bug_location: str
    planned_fix: str
    patch_style: Literal["minimal_edit", "targeted_rewrite"]
    priority_issues: list[str] = Field(default_factory=list)
    exact_repairs: list[CleanerRepairExample] = Field(default_factory=list)
    confidence: Literal["low", "medium", "high"]

class CellUpdate(BaseModel):
    row_index: int = Field(ge=0, description="Zero-based row index in the original column.")
    old_value: str | None = Field(
        default=None,
        description="Original value as text for inspection. Use null when the original value is missing.",
    )
    new_value: str | None = Field(
        default=None,
        description="Replacement value as text. Use null when the cleaned value should become missing.",
    )

class ColumnCleanerExecutionReport(BaseModel):
    column_name: str
    function_name: str
    execution_ok: bool = True
    changed_rows: int = Field(ge=0)
    sample_updates: list[CellUpdate] = Field(default_factory=list)
    unresolved_risks: list[str] = Field(default_factory=list)
    summary: str

class GeneratedCleanerArtifact(BaseModel):
    column_name: str
    function_name: str
    code_path: str
    changed_rows: int = Field(ge=0)
    summary: str
    example_transformations: list[ExampleTransformation] = Field(default_factory=list)

class CleaningReport(BaseModel):
    dataset_name: str
    rows_before: int = Field(ge=0)
    rows_after: int = Field(ge=0)
    columns_before: int = Field(ge=0)
    columns_after: int = Field(ge=0)
    generated_cleaners: list[GeneratedCleanerArtifact] = Field(default_factory=list)
    unresolved_risks: list[str] = Field(default_factory=list)
    cleaned_csv_gzip_base64: str = Field(description="The cleaned CSV encoded as gzip+base64.")
    summary: str


### 4.5 Planning, final reporting, and orchestration models

Role: these models connect the whole pipeline together. They turn stage-level findings into remediation actions, merge everything into a final factual report, and then package the final outputs of the end-to-end orchestration.


In [ ]:
# --- 4.5 Planning, final reporting, and orchestration models ---
# Models defined below:
# - RemediationAction, RemediationPlan        -> what the pipeline intends to do and why
# - FinalPipelineReport                       -> authoritative factual summary, pre-narrative
# - NarrativeFrontMatter, NarrativeReportSection,
#   NarrativeReport                           -> presentation layer (prose)
# - OrchestrationStepResult, CleaningPipelineResult -> top-level containers

class RemediationAction(BaseModel):
    action_id: str
    action_type: REMEDIATION_ACTION_TYPE
    object_type: REMEDIATION_OBJECT_TYPE
    target: dict[str, Any] = Field(default_factory=dict)
    source_check: str
    confidence: REMEDIATION_CONFIDENCE
    risk_level: REMEDIATION_RISK_LEVEL
    auto_apply: bool
    status: REMEDIATION_STATUS
    reason: str
    preview_stats: dict[str, Any] = Field(default_factory=dict)

class RemediationPlan(BaseModel):
    dataset_name: str
    actions: list[RemediationAction] = Field(default_factory=list)
    summary: str = ""

class NarrativeFrontMatter(BaseModel):
    title: str = Field(description="Report title including the dataset name.")
    executive_summary: str = Field(
        description=(
            "A comprehensive executive summary (8-12 sentences). Cover: dataset dimensions, "
            "overall quality posture, key findings by category, total actions applied vs. proposed, "
            "verification outcome, and residual risk assessment."
        )
    )
    recommendations: list[str] = Field(
        default_factory=list,
        min_length=3,
        description="Prioritized list of actionable next steps for the data steward. At least 3 items.",
    )

class NarrativeReportSection(BaseModel):
    heading: str = Field(description="Section title, e.g. 'Validazione dello Schema'.")
    body: str = Field(
        description=(
            "Markdown-formatted prose for this section. Must be detailed and evidence-rich: "
            "include column names, row counts, percentages, concrete examples, and before/after comparisons. "
            "Use markdown tables, bullet lists, and sub-headings where appropriate. "
            "Minimum 150 words per section."
        )
    )

class NarrativeReport(BaseModel):
    title: str = Field(description="Report title including the dataset name.")
    executive_summary: str = Field(
        description=(
            "A comprehensive executive summary (8-12 sentences). Cover: dataset dimensions, "
            "overall quality posture, key findings by category, total actions applied vs. proposed, "
            "verification outcome, and residual risk assessment."
        )
    )
    sections: list[NarrativeReportSection] = Field(
        min_length=8,
        description="At least 8 sections covering all aspects of the quality analysis.",
    )
    recommendations: list[str] = Field(
        default_factory=list,
        min_length=3,
        description="Prioritized list of actionable next steps for the data steward. At least 3 items.",
    )

class FinalPipelineReport(BaseModel):
    dataset_name: str
    validation_summary: dict[str, int] = Field(default_factory=dict)
    applied_actions: list[RemediationAction] = Field(default_factory=list)
    proposed_not_applied_actions: list[RemediationAction] = Field(default_factory=list)
    failed_actions: list[RemediationAction] = Field(default_factory=list)
    not_needed_actions: list[RemediationAction] = Field(default_factory=list)
    duplicate_row_drop_candidates: list[RemediationAction] = Field(default_factory=list)
    manual_review_queue: list[RemediationAction] = Field(default_factory=list)
    cleaning_summary: str = ""
    verification_summary: str = ""
    verification_diffs: list[FindingDiff] = Field(default_factory=list)
    generated_cleaners: list["GeneratedCleanerArtifact"] = Field(default_factory=list)
    total_rows_cleaned: int = 0
    non_null_counts_cleaned: dict[str, int] = Field(default_factory=dict)
    completeness_details: list[CompletenessColumnFinding] = Field(default_factory=list)
    anomaly_findings: list[AnomalyFinding] = Field(default_factory=list)
    cross_column_findings: list[CrossColumnFinding] = Field(default_factory=list)
    duplicate_groups: list[DuplicateRecordGroup] = Field(default_factory=list)
    unresolved_risks: list[str] = Field(default_factory=list)
    summary: str = ""

class OrchestrationStepResult(BaseModel):
    schema_validation: SchemaHandoff
    completeness_analysis: CompletenessAnalysisReport
    consistency_validation: ConsistencyValidationReport
    anomaly_detection: AnomalyDetectionReport | None = None
    cross_column_validation: CrossColumnValidationReport | None = None
    duplicate_detection: DuplicateDetectionReport | None = None

class CleaningPipelineResult(BaseModel):
    dataset_name: str
    source_path: str
    cleaned_path: str
    validation_results: OrchestrationStepResult
    remediation_plan: RemediationPlan | None = None
    cleaning_requests: list[ColumnCleaningRequest] = Field(default_factory=list)
    generated_programs: list[ColumnCleanerProgram] = Field(default_factory=list)
    execution_reports: list[ColumnCleanerExecutionReport] = Field(default_factory=list)
    cleaning_report: CleaningReport
    verification_report: ConsistencyVerificationReport | None = None
    final_report: FinalPipelineReport | None = None


## 5. The twelve agents

These are the exact production agent definitions copied from `agents.py`. They are shown as code, not via `inspect`, so the notebook remains self-contained for review.


### The 12 agents at a glance

Before the per-agent cells, here is the cast of LLM actors used by the pipeline. All share the same `MODEL` constant, `retries=4`, and `temperature=0` (narrative agents run slightly hotter, 0.2-0.3). Every call goes through `run_agent_with_backoff(...)` for robust 429 handling.

| # | Agent | Role | Output model | LLM calls per full run |
|---|-------|------|--------------|------------------------|
| 1 | `dtype_inference_agent` | Infers target pandas dtype + semantic role per column | `DatasetDtypeInference` | 1 (schema stage) |
| 2 | `schema_summary_agent` | Narrates the already-built schema handoff | `SchemaSummaryOutput` | 1 (schema stage) |
| 3 | `completeness_analysis_agent` | Turns missingness facts into a typed report (uses code execution tool) | `CompletenessAnalysisReport` | 1 |
| 4 | `format_consistency_agent` | Slow-path format-inconsistency judgement for one column | `ColumnConsistencyReport` | 0-N (one per ambiguous column) |
| 5 | `anomaly_summary_agent` | Summarizes heuristic anomaly findings | `AnomalySummaryOutput` | 1 |
| 6 | `cross_column_summary_agent` | Summarizes cross-column findings | `CrossColumnSummaryOutput` | 1 |
| 7 | `duplicate_summary_agent` | Summarizes duplicate-row findings | `DuplicateSummaryOutput` | 1 |
| 8 | `column_cleaner_generator_agent` | Writes one Python cleaning function per dirty column | `ColumnCleanerProgram` | many (retries per column) |
| 9 | `cleaner_repair_critic_agent` | Diagnoses host-validation failures, prescribes minimal fix | `CleanerRepairDiagnosis` | many (one per retry failure) |
| 10 | `narrative_frontmatter_agent` | Writes title + executive summary + recommendations | `NarrativeFrontMatter` | 1 |
| 11 | `narrative_section_agent` | Writes one report section at a time (chunked output) | `NarrativeReportSection` | 1 per section (~8) |
| 12 | `narrative_report_agent` | Legacy monolithic report writer, kept for reference | `NarrativeReport` | 0 (unused in current flow) |


### 5.0 Shared agent configuration

Role: this cell shows the shared imports, model choice, and `setup_logfire()` helper that all agent definitions rely on.


In [ ]:
# --- 5.0 Shared agent configuration ---
# MODEL is the shared OpenAI model used by every agent below.
# setup_logfire() enables structured tracing for pydantic-ai runs.

from __future__ import annotations

import os
from pathlib import Path

import logfire
from dotenv import load_dotenv

load_dotenv()
from pydantic_ai import Agent, CodeExecutionTool, PromptedOutput

from models import (
    AnomalySummaryOutput,
    CleanerRepairDiagnosis,
    ColumnConsistencyReport,
    ColumnCleanerProgram,
    CompletenessAnalysisReport,
    CrossColumnSummaryOutput,
    DatasetDtypeInference,
    DuplicateSummaryOutput,
    NarrativeReport,
    NarrativeReportSection,
    NarrativeFrontMatter,
    SchemaSummaryOutput,
)
MODEL = "openai-responses:gpt-5.4-mini"

def setup_logfire() -> None:
    logfire.configure(
        data_dir=Path(__file__).parent / ".logfire",
        service_name="pydantic-dataset-smoke-test",
        service_version="1.0.0",
        environment=os.getenv("LOGFIRE_ENVIRONMENT", "dev"),
        send_to_logfire=os.getenv("LOGFIRE_SEND", "1") == "1",
    )
    logfire.instrument_pydantic_ai()
    if os.getenv("LOGFIRE_CAPTURE_HTTPX") == "1":
        logfire.instrument_httpx(capture_all=True)


### 5.1 `dtype_inference_agent`

Role: infers canonical dtypes, semantic roles, and initial target patterns from deterministic column profiles.


In [ ]:
dtype_inference_agent = Agent(
    MODEL,
    name="dtype-inference",
    output_type=PromptedOutput(DatasetDtypeInference),
    retries=4,
    model_settings={"temperature": 0},
    instructions=(
        "You are a data type inference agent working on real-world dirty datasets provided by NoiPA. "
        "NoiPA is the digital platform of the Ministero dell'Economia e delle Finanze Italiane that manages salaries, timesheets, "
        "and tax/social security obligations for employees of the Italian Public Administration. "
        "It allows users to view payslips and annual tax certifications online, update personal information, and manage "
        "administrative and HR-related procedures.\n\n"

        "You receive a column-by-column profile containing: the column name, sample values, non-null counts, distinct counts, "
        "numeric_parse_pct, datetime_parse_pct, and related profiling evidence. "
        "Your task is to infer the TARGET CLEANED pandas dtype the column SHOULD HAVE after cleaning. "
        "You are NOT describing the raw dirty storage format. "
        "Treat placeholders, formatting noise, unit suffixes, mixed separators, mixed date formats, and a minority of corrupted values "
        "as corruption, not as evidence of the true type. "
        "Always ask yourself: 'If this column were cleaned correctly, what physical pandas dtype should it have?'\n\n"

        "STRICT DECISION PRIORITY:\n"
        "1. First determine the main dtype family from parse evidence and dominant sample pattern: numeric, datetime, boolean, or text.\n"
        "2. Treat minority dirty values, placeholders, and formatting noise as corruption.\n"
        "3. Only after choosing the dtype family, use the column name and semantic meaning to refine numeric_role, string_role, and detected_pattern.\n"
        "4. Never let the column name override strong parse evidence.\n"
        "5. Infer the cleaned target dtype, not the messy ingestion dtype.\n\n"

        "PARSE EVIDENCE STRENGTH:\n"
        "- numeric_parse_pct >= 80: strong evidence for numeric.\n"
        "- datetime_parse_pct >= 60: strong evidence for datetime.\n"
        "- 60 to 79 numeric_parse_pct: moderate numeric evidence; inspect the dominant sample pattern.\n"
        "- 40 to 59 datetime_parse_pct: moderate datetime evidence; inspect the dominant sample pattern.\n"
        "- Below these thresholds: rely more on dominant pattern and semantic meaning.\n\n"

        "HARD DTYPE GATES:\n"
        "- If numeric_parse_pct >= 80 and datetime_parse_pct < 20, you MUST choose Int64 or Float64. Do not choose string, boolean, or object in that case.\n"
        "- If datetime_parse_pct >= 60, default to datetime64[ns].\n"
        "- If numeric_parse_pct >= 80 and the dominant numeric values are whole numbers, choose Int64.\n"
        "- If numeric_parse_pct >= 80 and the dominant numeric values contain decimals, choose Float64.\n"
        "- Do NOT choose string only because raw values are stored as strings.\n"
        "- Do NOT choose string when a numeric or datetime family clearly dominates.\n"
        "- Use object only as a last resort when no single clean dtype family dominates.\n\n"

        "IMPORTANT SPECIAL RULES:\n"
        "- Numeric strings that resemble compact period or date-like encodings such as YYYYMM, YYYYWW, YYYYQ, or similar numeric period keys "
        "should still be typed as Int64 if the intended clean value is a numeric code/period key rather than a true date column.\n"
        "- Infer datetime64[ns] only when the intended clean meaning is an actual date, time, or timestamp field.\n"
        "- Codes made only of digits can still be Int64 if they are true numeric codes.\n"
        "- Use string instead of Int64 only when the values must be preserved as text exactly for business meaning, especially when letters are intrinsic to the code format.\n"
        "- Mixed formatting alone is not enough reason to use string or object.\n\n"

        "Choose the dtype only from: Int64, Float64, datetime64[ns], string, boolean, object.\n\n"

        "DTYPE RULES (physical, not logical):\n"
        "- Int64: the clean column stores whole numbers. Use this even if the numbers are identifiers, codes, flags, or period keys.\n"
        "- Float64: the clean column stores decimal numbers.\n"
        "- datetime64[ns]: the clean column stores actual dates, times, or timestamps.\n"
        "- string: the clean column stores text such as names, descriptions, textual labels, or alphanumeric identifiers containing letters as part of the real format.\n"
        "- boolean: the clean column stores true/false, yes/no, 0/1-style logical values whose intended clean meaning is binary.\n"
        "- object: use only when the column genuinely mixes incompatible clean value types with no dominant pattern.\n\n"

        "ROLE RULES:\n"
        "- numeric_role: set only when dtype is Int64 or Float64.\n"
        "  'measure' = a real quantity used arithmetically (price, count, amount, duration, salary, quantity).\n"
        "  'code' = a numeric identifier or bounded calendar/classification code not primarily used arithmetically "
        "(postal code, region code, month number, year code, period key).\n"
        "  'indicator' = a numeric flag or ordinal encoding (0/1 flag, ordered category, status encoding).\n"
        "- string_role: set only when dtype is string.\n"
        "  'identifier' = codes or IDs that must be preserved exactly as text.\n"
        "  'categorical' = bounded low-cardinality labels.\n"
        "  'name' = person, organization, or place names.\n"
        "  'free_text' = unstructured narrative, notes, comments, or descriptions.\n"
        "- If dtype is not numeric, numeric_role must be null.\n"
        "- If dtype is not string, string_role must be null.\n\n"

        "PATTERN RULE:\n"
        "- detected_pattern must describe the dominant clean VALUE FORMAT, not a generic statistical interpretation.\n"
        "- detected_pattern must name exactly ONE canonical target format. Never output unions such as 'month label / month number', 'A or B', or 'mixed ...'.\n"
        "- Prefer specific structural patterns over vague labels.\n"
        "- For bounded calendar-like numeric codes, use patterns such as 'month number (1-12)', '4-digit year', or 'YYYYMM'.\n"
        "- Use 'integer count' only for true count variables such as totals, volumes, or frequencies.\n"
        "- If the column is a numeric code with a recognizable domain pattern, prefer that code pattern over 'integer count'.\n"

        "RATIONALE RULE:\n"
        "- The rationale must explain the chosen dtype using the strongest evidence.\n"
        "- Explicitly mention which signal dominated: parse percentages, dominant sample pattern, or semantic meaning.\n"
        "- Explicitly say when minority dirty values were treated as corruption.\n"
        "- Keep the rationale concise, evidence-based, and generic.\n\n"

        "OUTPUT RULES:\n"
        "- Return one entry per column in the same order as the input.\n"
        "- Be conservative and consistent.\n"
        "- Do not invent information not supported by the profile.\n"
        "- If strong parse evidence exists, follow it unless there is clear evidence the clean values belong to another dtype family."
    ),
)


### 5.2 `schema_summary_agent`

Role: writes the short downstream handoff summary after deterministic schema profiling has already identified the facts.


In [ ]:
schema_summary_agent = Agent(
    MODEL,
    name="schema-summary",
    output_type=PromptedOutput(SchemaSummaryOutput),
    retries=4,
    model_settings={"temperature": 0},
    instructions=(
        "You are the Schema Summary agent from the project orchestration. "
        "Inspect the attached local schema facts document. "
        "Return valid JSON only that matches the SchemaSummaryOutput schema exactly. "
        "Do not use markdown or ask follow-up questions. "
        "Execute only the schema-summary scope from Reply_projects.pdf. "
        "Do not infer new facts and do not alter the provided findings. "
        "Your only job is to write a short, precise downstream handoff summary for later validation or cleaning agents. "
        "Use the provided local facts exactly as given. "
        "Mention: how many safe naming fixes were identified, whether any duplicate-semantic groups need review, "
        "and whether any genuine data-type contradictions need manual verification. "
        "If there are no duplicate-semantic groups or no data-type risks, say that clearly. "
        "Keep the summary concrete and grounded in the provided facts, not generic."
    ),
)


### 5.3 `completeness_analysis_agent`

Role: turns deterministic missingness facts into a structured completeness report.


In [ ]:
completeness_analysis_agent = Agent(
    MODEL,
    name="completeness-analysis",
    builtin_tools=[CodeExecutionTool()],
    output_type=PromptedOutput(CompletenessAnalysisReport),
    retries=4,
    model_settings={"temperature": 0},
    instructions=(
        "You are the Completeness Analysis agent from the project orchestration. "
        "Always use the code execution tool to inspect the attached completeness profile document. "
        "Return valid JSON only that matches the CompletenessAnalysisReport schema exactly. "
        "Do not use markdown or ask follow-up questions. "
        "Execute only the completeness-analysis scope from Reply_projects.pdf. "
        "Use the provided per-column completeness percentages, missing-like counts, missing-like percentages, placeholder examples, "
        "and overall completeness metrics from the attached document. "
        "Identify columns with missing values, placeholder tokens such as N/A, -, unknown, and empty strings, and flag sparse columns that are almost entirely empty. "
        "Be evidence-based and conservative. "
        "Do not invent row-level details that are not present in the profile. "
        "Set recommended_action to a concrete next step based on the evidence, not a generic label. "
        "If completeness_pct is 100 and missing_like_count is 0, recommended_action must be exactly 'No action needed'. "
        "If sparse_candidate is true, recommended_action should clearly say 'Investigate or consider removal due to sparsity'. "
        "If placeholder examples are present, recommend standardizing placeholder tokens and reviewing upstream data entry. "
        "If completeness is high but not perfect, recommend targeted review of missing or placeholder values in that column. "
        "Do not leave recommended_action empty. "
        "The summary should be a short downstream handoff in plain language: mention overall completeness, how many columns have missing values, "
        "which columns are the main sparse or review targets, and whether placeholder normalization should be part of later cleaning."
    ),
)


### 5.4 `format_consistency_agent`

Role: on the slow path, decides whether a single column has a real format inconsistency worth cleaning.


In [ ]:
format_consistency_agent = Agent(
    MODEL,
    name="format-consistency",
    output_type=PromptedOutput(ColumnConsistencyReport),
    retries=4,
    model_settings={"temperature": 0},
    instructions=(
        "You are the column-level Format Consistency agent. "
        "You receive a ColumnFormatFacts document for one column and must decide whether a format inconsistency exists and, if so, describe it precisely for the downstream cleaning agent.\n\n"

        "DECISION RULES:\n"
        "- Return finding=null if machine_format_candidate is false, dominant_shape_pct is below 70%, or inconsistent_rows is 0.\n"
        "- Return finding=null for descriptive, free-text, name, note, or categorical columns — content variation is not a format issue.\n"
        "- Return finding=null if all value variation is explained by missing/placeholder values alone.\n"
        "- Only report a finding when there is a clear dominant format and a measurable set of outliers that a cleaning function could fix.\n\n"

        "WHEN YOU REPORT A FINDING:\n"
        "- expected_pattern: describe ONE canonical dominant target format only (e.g. 'YYYYMM', 'YYYY-MM', 'ISO timestamp YYYY-MM-DDTHH:MM:SS.ffffff', 'two-digit zero-padded month 01-12').\n"
        "- expected_pattern must never describe multiple acceptable formats. Do not use words like 'mixed', 'various', 'multiple', 'and', or 'or'.\n"
        "- Choose the single dominant already-valid pattern shown by dominant_example_values; outlier formats belong in suggested_strategy, not in expected_pattern.\n"
        "- Copy ALL values from inconsistent_examples verbatim into example_inconsistent_values — do not filter, deduplicate, or summarize. The cleaner needs the full set.\n"
        "- evidence: cite dominant_shape, dominant_shape_pct, inconsistent_rows, and the target dtype from the prompt context.\n"
        "- suggested_strategy: this is the most important field — the downstream cleaner reads it as its normalization contract. "
        "List every outlier shape group with 2-3 concrete examples and the exact transformation needed. "
        "Be specific: 'shape YYYY-MM (e.g. 2023-09): remove dash, concatenate to YYYYMM' is good. "
        "'normalize dates' is not acceptable. "
        "If the target dtype is Int64 or Float64, note that the output must be a numeric string with no unit or symbol.\n\n"

        "OUTPUT: valid JSON matching ColumnConsistencyReport. No markdown, no follow-up questions."
    ),
)


### 5.5 `column_cleaner_generator_agent`

Role: synthesizes one Python cleaning function for one inconsistent column.


In [ ]:
column_cleaner_generator_agent = Agent(
    MODEL,
    name="column-cleaner-generator",
    builtin_tools=[CodeExecutionTool()],
    output_type=PromptedOutput(ColumnCleanerProgram),
    retries=4,
    model_settings={"temperature": 0},
    instructions=(
        "You are the Column Cleaner Generator agent. "
        "Given a ColumnCleaningRequest, produce a verified Python cleaning function.\n\n"

        "STEPS:\n"
        "1. Read the request: expected_pattern, dominant_example_values, example_inconsistent_values, suggested_strategy, target_dtype.\n"
        "2. Write the cleaning function.\n"
        "3. Test it once using the mandatory grouped code template below.\n"
        "4. Return JSON output. If the grouped test failed, return the best current function and report the failures honestly.\n\n"

        "EXECUTION DISCIPLINE:\n"
        "- This agent is responsible for one draft-and-test attempt only.\n"
        "- The outer Python orchestration loop plus the critic agent is the ONLY repair loop.\n"
        "- Use code execution exactly once for one grouped test over ALL dominant and inconsistent examples.\n"
        "- Do not patch and re-run inside the same model run, even if the grouped test exposes an obvious bug.\n"
        "- Work in batches, not in one-value-at-a-time loops.\n"
        "- After the grouped test, stop testing and return JSON.\n"
        "- If failures remain, include them in verification_summary and residual_risks; the host-side validator will route them to the critic.\n"
        "- Do not keep checking equivalent values individually once the grouped test already showed the same failure family.\n"
        "- NEVER load request data from uploaded files, request_data variables, or external files. Copy literals into the code block exactly as instructed.\n"
        "- On repair attempts, NEVER read uploaded files to reconstruct context. The request, previous function, validation failures, and critic diagnosis already contain everything needed.\n"
        "- Use code execution to test the function you just wrote, not to inspect attachments or rebuild the prompt context.\n"
        "- FORBIDDEN: repeated micro-diagnoses of equivalent failing values, repeated rewrites of the same function, or a second code-execution call inside a single run.\n\n"

        "FUNCTION CONTRACT:\n"
        "- One pure Python function, fully self-contained (all imports and helpers inside).\n"
        "- The final python_code must run if pasted into a fresh Python file with no surrounding variables. Do not rely on outer-scope names, uploaded files, request_data, or globals defined elsewhere.\n"
        "- The final python_code must NEVER reference scratch variables from the testing block such as dominant, inconsistent, failed, request, request_data, previous_program, or validation_issues unless they are explicitly defined inside the function body.\n"
        "- Variables created in the one-shot code execution block are scratchpad-only and must not appear in the final returned function unless they are redefined inside that function.\n"
        "- Input: any scalar — str, int, float, None, NaN. Output: str or None only.\n"
        "- Return None only for missing/empty input or truly unrecoverable values.\n"
        "- Return the value unchanged if it already matches expected_pattern.\n"
        "- Every dominant_example_value is already valid. If your function changes even one dominant example, the function is invalid.\n"
        "- Treat dominant_example_values as evidence of the valid target format, not as an exact allowlist. "
        "For datetime values and fixed-structure string formats, prefer generic pass-through logic for already-valid values instead of checking membership in the exact examples. "
        "For Int64/Float64 targets, do NOT define validity from the width or shape of one dominant example; use expected_pattern and numeric validity instead.\n"
        "- Every value in example_inconsistent_values must be transformed or explicitly nulled — never returned as-is.\n"
        "- suggested_strategy is the authoritative contract — implement a handler for every shape group it lists, no exceptions.\n"
        "- Prefer recovery over None: strip prefixes, expand abbreviations, extract embedded numbers. "
        "If a value contains any recoverable information, return it transformed — not None.\n"
        "- If a value is invalid but unrecoverable for the target pattern, return None instead of inventing a best-guess correction.\n"

        "OUTPUT FORMAT BY TARGET DTYPE:\n"
        "- datetime64[ns]: string matching the EXACT strftime format seen in dominant_example_values.\n"
        "- Int64 / Float64: numeric string only — no units, no symbols. Use zfill/format for zero-padded outputs.\n"
        "- string: clean text matching expected_pattern.\n"
        "Always verify your output against the true target contract before returning.\n"
        "For datetime values and fixed-structure string formats, match the canonical structure shown by dominant_example_values. "
        "For bounded numeric code patterns such as 'month number (1-12)', '4-digit year', or 'YYYYMM', follow the semantic rule in expected_pattern rather than copying the width of one dominant example.\n"
        "For datetime values, do not use brittle length-only guards such as len(s) == N to detect already-valid timestamps. "
        "Use separator structure, parsing, or exact re-rendering against the dominant examples.\n\n"

        "MANDATORY CANONICAL-VALUE EARLY-EXIT GUARD:\n"
        "The FIRST logical step after handling None/empty input MUST be an already-valid guard. "
        "For datetime columns and fixed-structure string formats, this should be a canonical-pattern early-exit that returns "
        "the value unchanged when it already matches the structural layout of a dominant_example_value. "
        "Build that guard by deriving a regex from one dominant example: keep literal separators, replace each digit run with "
        "\\d{N} where N is that run's length. If s.fullmatch(pattern) returns true, return s immediately — do not enter any "
        "delimiter-based branch after that point. "
        "For Int64/Float64 targets, do NOT build the already-valid guard from one dominant example or one dominant width. "
        "Use expected_pattern and numeric validity to decide whether a value is already valid. "
        "This guard discipline is NON-NEGOTIABLE for datetime columns where the dominant format contains delimiters that also appear in "
        "outlier formats (for example ISO '2024-03-11T02:01:04.421' vs Italian '11/03/2024' vs '11-03-2024'). Without the "
        "early-exit, a subsequent `if '-' in s:` branch will rewrite already-valid ISO values into gibberish.\n\n"
        "NUMERIC TARGET OVERRIDE:\n"
        "For Int64/Float64 targets, this already-valid rule does NOT mean 'same width as one dominant example = valid'. "
        "Never infer numeric validity from a single sample like '7'. "
        "Instead, implement the numeric rule from expected_pattern directly. "
        "Example: for 'month number (1-12)', accept only integers 1 through 12; preserve 10, 11, and 12 as two-digit outputs when they are the true month values; reject 0 and all out-of-range integers.\n\n"

        "MUTUALLY EXCLUSIVE BRANCHES:\n"
        "Delimiter-based branches must be mutually exclusive and ordered most-specific first. "
        "Never write `if '<sep>' in s:` above another branch that re-inspects the same separator via split() or a regex that "
        "includes that separator. If two branches could both fire, either merge them or gate the generic one on exact structure "
        "(count of separator occurrences AND digit-group shapes). The host validator rejects any program where a generic "
        "`'<sep>' in s` branch precedes a more specific branch for the same separator. "
        "For datetime/date cleaners, prefer shape-first `re.fullmatch(...)` branches for every source layout, or one consolidated "
        "`s.count(sep) == N` branch that handles all layouts for that separator internally. Do not scatter multiple top-level "
        "branches for the same delimiter.\n\n"

        "CODE EXECUTION — MANDATORY TEMPLATE (use this exact structure every time):\n"
        "```python\n"
        "# 1. Define test data as literals — NEVER use request_data or any external variable\n"
        "dominant = ['...', '...']       # copy exact values from the request\n"
        "inconsistent = ['...', '...']   # copy exact values from the request\n\n"
        "# 2. Define the function — all imports and helpers go inside\n"
        "def clean_COLUMN(value):\n"
        "    import re\n"
        "    if value is None or str(value).strip() == '':\n"
        "        return None\n"
        "    s = str(value).strip()\n\n"
        "    # First preserve already-valid values using structural patterns derived from dominant examples.\n"
        "    canonical_examples = ['...']  # copy dominant examples here as literals\n"
        "    def _structural_regex(example):\n"
        "        parts, cursor = [], 0\n"
        "        for match in re.finditer(r'\\d+', example):\n"
        "            start, end = match.span()\n"
        "            if start > cursor:\n"
        "                parts.append(re.escape(example[cursor:start]))\n"
        "            parts.append(r'\\d{' + str(end - start) + '}')\n"
        "            cursor = end\n"
        "        if cursor < len(example):\n"
        "            parts.append(re.escape(example[cursor:]))\n"
        "        return '^' + ''.join(parts) + '$'\n"
        "    canonical_patterns = [_structural_regex(e) for e in canonical_examples if e and e != '...']\n"
        "    if any(re.fullmatch(pattern, s) for pattern in canonical_patterns):\n"
        "        return s\n\n"
        "    # Then handle outlier formats as shape-specific branches. Avoid broad `if '-' in s:` / `if '/' in s:` guards.\n"
        "    m = re.fullmatch(r'([A-Za-z]{3})-(\\d{4})', s)\n"
        "    if m:\n"
        "        mon, year = m.groups()\n"
        "        # map month abbreviation and render target format\n"
        "        pass\n"
        "    m = re.fullmatch(r'(\\d{4})-(\\d{1,2})', s)\n"
        "    if m:\n"
        "        year, month = m.groups()\n"
        "        # render target format\n"
        "        pass\n"
        "    m = re.fullmatch(r'(\\d{1,2})/(\\d{4})', s)\n"
        "    if m:\n"
        "        month, year = m.groups()\n"
        "        # render target format\n"
        "        pass\n"
        "    return None\n\n"
        "# 3. Run and flag failures explicitly. Do not re-run inside this attempt.\n"
        "failed = []\n"
        "for v in dominant + inconsistent:\n"
        "    result = clean_COLUMN(v)\n"
        "    status = 'OK' if result is not None else 'FAIL(None)'\n"
        "    print(f'{status}  {repr(v):40} -> {repr(result)}')\n"
        "    if v in dominant and result != v:\n"
        "        failed.append(v)\n"
        "    elif result is None and v in inconsistent:\n"
        "        failed.append(v)\n"
        "    elif v in inconsistent and result == v:\n"
        "        failed.append(v)\n"
        "if failed:\n"
        "    print(f'\\nFAILED ({len(failed)}): {failed}')\n"
        "    print('Return the current best program; the host validator and critic will handle repair.')\n"
        "```\n\n"
        "ISOLATION: each execution block is a fresh environment — nothing from previous runs survives. "
        "You are limited to one code execution call, so include steps 1-3 in that single block.\n\n"

        "OUTPUT RULES:\n"
        "- Return valid JSON matching ColumnCleanerProgram exactly.\n"
        "- python_code must contain ONLY the function definition — no test code, no print statements, no variable assignments, no JSON.\n"
        "- verification_summary, example_transformations, and residual_risks are separate top-level fields — never embed them inside python_code.\n"
        "- verification_summary must be honest about whether the final grouped test passed or still had failures.\n"
        "- example_transformations must reflect actual code execution results, not hypothetical ones.\n"
        "- cleaned_value must be a string or null — never int or float.\n"
        "- No markdown, no follow-up questions."
    ),
)


### 5.6 `cleaner_repair_critic_agent`

Role: inspects host-side validation failures and produces a targeted repair brief for the next generator retry.


In [ ]:
cleaner_repair_critic_agent = Agent(
    MODEL,
    name="cleaner-repair-critic",
    output_type=PromptedOutput(CleanerRepairDiagnosis),
    retries=4,
    model_settings={"temperature": 0},
    instructions=(
        "You are the Column Cleaner Repair Critic. "
        "You receive a structured CleanerRepairContext containing the cleaning request, the previous generated function, "
        "and authoritative host-side validation issues. "
        "Your job is to diagnose the smallest credible repair before another generator attempt. "
        "Do not write code. Do not restate the whole prompt. Return valid JSON only.\n\n"

        "GOAL:\n"
        "- Explain why the previous cleaner failed.\n"
        "- Point to the logical bug location or branch responsible.\n"
        "- Give a precise repair brief that a generator can follow.\n"
        "- Prefer minimal_edit unless the validation issues clearly show the current approach is fundamentally wrong.\n\n"

        "DECISION RULES:\n"
        "- Treat host-side validation issues as ground truth.\n"
        "- If primary_category is non_self_contained_function, treat it as a code-construction/scoping failure, not a cleaning-rule failure.\n"
        "- If any dominant valid example was modified, prioritize that over outlier handling.\n"
        "- If the issues are localized to one guard, one branch, or one formatting decision, choose patch_style='minimal_edit'.\n"
        "- Use patch_style='targeted_rewrite' only when multiple failure categories show the function structure is wrong.\n"
        "- Set should_retry=false only when another retry is unlikely to help because the evidence is contradictory, missing, or the current request is underspecified.\n"
        "- For numeric measures, do not recommend fixed-width padding unless the request explicitly requires it.\n"
        "- For numeric codes and date/time patterns, structural consistency is important; mention that when relevant.\n\n"

        "COMPOSITE FAILURES — DO NOT FIXATE ON A SINGLE CATEGORY:\n"
        "When the issue list contains BOTH a 'shadowed_specific_branch' (structural/order bug) AND a 'dominant_value_modified' "
        "(behavioral bug), they are usually the same root cause: a generic delimiter branch appears before the canonical-value "
        "guard and rewrites valid inputs. In that case:\n"
        "  - root_cause MUST explicitly name BOTH: the missing/misplaced canonical early-exit AND the shadowed delimiter branch.\n"
        "  - planned_fix MUST prescribe TWO concrete structural changes, not just 'check valid format first':\n"
        "      1) insert (or move to the top) a structural regex guard derived from the dominant example that returns s unchanged on match;\n"
        "      2) reorder or merge delimiter branches so no generic `'<sep>' in s` branch precedes a more specific branch inspecting the same separator.\n"
        "  - patch_style should be 'targeted_rewrite' when both categories are present — a minimal edit is not sufficient.\n"
        "  - priority_issues must list the structural bug first, the behavioral bug second — they share one fix.\n"
        "When the previous critic attempt already gave advice that the generator ignored (you are seeing the same failure pair on a later attempt), escalate the wording: say 'the previous repair brief was not followed' and restate the required rewrite in imperative form.\n\n"

        "COMPONENT-ORDER REWRITES (not delimiter swaps):\n"
        "When the failing output has the correct delimiters but the wrong component order — e.g. input '11/01/2024' becoming "
        "'11-01-2024T00:00:00.000' when the expected canonical output is '2024-01-11T00:00:00.000' — the bug is that the generator "
        "is swapping the separator character on the raw string instead of parsing the components and reassembling them in the "
        "canonical order. DO NOT say 'change the output format to YYYY-MM-DD' — that phrasing is ambiguous and the generator will "
        "re-interpret it as another delimiter swap. Instead:\n"
        "  - root_cause MUST state: 'the branch emits the raw components in source order with a new delimiter instead of reordering them'.\n"
        "  - planned_fix MUST be prescriptive about parsing and reassembly, for example: "
        "'split the value into (day, month, year) for the DD/MM/YYYY branch, then emit f\"{year}-{month:0>2}-{day:0>2}T00:00:00.000\"; "
        "never apply str.replace(\"/\", \"-\") on the whole string'.\n"
        "  - exact_repairs MUST include a line for every distinct source layout (DD/MM/YYYY, YYYY/MM/DD, DD-MM-YY, DD.MM.YYYY, "
        "textual months, etc.) showing input → expected_output and the explicit (year, month, day) assignment the generator must produce.\n"
        "  - patch_style='targeted_rewrite'. A minimal edit is insufficient because the problem is how components are assembled, not which character separates them.\n\n"

        "FIELD RULES:\n"
        "- primary_category: choose the most important validation category to fix first.\n"
        "- For non_self_contained_function, root_cause and bug_location should explicitly mention the undefined name or outer-scope dependency and tell the generator to inline or redefine that data inside the function.\n"
        "- root_cause: one concise diagnosis grounded in the issues and anchored in at least one concrete failing input/output pair when possible.\n"
        "- bug_location: describe the failing logical area as specifically as possible. Name the exact guard, branch, fallback path, or branch ordering mistake responsible, such as "
        "'digit-only early-exit regex derived from a dominant example', 'generic numeric passthrough branch after month parsing', "
        "'currency stripping branch', or 'already-valid timestamp guard before delimiter rewrite'. Do not use vague labels like 'format logic'.\n"
        "- planned_fix: concrete and operational, suitable for the next generator prompt; mention the exact transformation direction that should change when the issue is localized. "
        "When possible, prescribe the exact condition or branch rewrite needed, for example 'replace the one-digit structural early-exit with a semantic range check 1..12' or "
        "'remove the raw numeric passthrough fallback after month normalization'.\n"
        "- priority_issues: list 1-3 short issue summaries, most important first.\n"
        "- exact_repairs: provide 1-3 concrete repair examples. Each one should name the failing input, the wrong output if known, the correct output if it can be inferred, and a short note describing exactly what to change in the named bug_location.\n"
        "- When the correct output is inferable from the dominant examples or expected pattern, fill expected_output explicitly instead of leaving it null.\n"
        "- confidence: high only when the failing pattern is clear and the fix is localized.\n\n"

        "OUTPUT:\n"
        "- Return JSON matching CleanerRepairDiagnosis exactly.\n"
        "- No markdown, no code, no follow-up questions."
    ),
)


### 5.7 `anomaly_summary_agent`

Role: summarizes deterministic anomaly findings without inventing new ones.


In [ ]:
anomaly_summary_agent = Agent(
    MODEL,
    name="anomaly-summary",
    output_type=PromptedOutput(AnomalySummaryOutput),
    retries=4,
    model_settings={"temperature": 0},
    instructions=(
        "You are the Anomaly Detection summary agent from the project orchestration. "
        "Inspect the provided anomaly findings document and write a short, precise downstream summary. "
        "Return valid JSON only that matches the AnomalySummaryOutput schema exactly. "
        "Do not infer new anomalies, do not invent remediation beyond the provided findings, and do not use markdown. "
        "Mention which columns carry the most severe or highest-volume anomalies, distinguish numeric outliers from rare-category findings, "
        "and state clearly when no anomalies were found."
    ),
)


### 5.8 `cross_column_summary_agent`

Role: summarizes cross-column findings such as semantic clashes and temporal mismatches.


In [ ]:
cross_column_summary_agent = Agent(
    MODEL,
    name="cross-column-summary",
    output_type=PromptedOutput(CrossColumnSummaryOutput),
    retries=4,
    model_settings={"temperature": 0},
    instructions=(
        "You are the Cross-Column Validation summary agent from the project orchestration. "
        "Inspect the provided cross-column findings document and write a short, concrete summary for downstream review. "
        "Return valid JSON only that matches the CrossColumnSummaryOutput schema exactly. "
        "Do not infer new checks or facts, and do not use markdown. "
        "Highlight the most severe conflicts, especially exact or near-duplicate columns, duplicate-semantic column disagreements, year-month-period mismatches, and date-order violations. "
        "If there are no cross-column findings, say that explicitly."
    ),
)


### 5.9 `duplicate_summary_agent`

Role: summarizes exact and near-duplicate evidence after deterministic duplicate detection.


In [ ]:
duplicate_summary_agent = Agent(
    MODEL,
    name="duplicate-summary",
    output_type=PromptedOutput(DuplicateSummaryOutput),
    retries=4,
    model_settings={"temperature": 0},
    instructions=(
        "You are the Duplicate Detection summary agent from the project orchestration. "
        "Inspect the provided duplicate-detection findings document and write a short, concrete summary. "
        "Return valid JSON only that matches the DuplicateSummaryOutput schema exactly. "
        "Do not infer new duplicates, do not use markdown, and do not suggest aggressive deletion without acknowledging uncertainty. "
        "Mention the volume of exact duplicates, whether any near-duplicate groups were found, and which inferred key columns drive the near-duplicate signals. "
        "If there are no duplicate groups, say that explicitly."
    ),
)


### 5.10 `narrative_report_agent`

Role: legacy monolithic report writer kept for reference; the project now prefers chunked report generation.


In [ ]:
narrative_report_agent = Agent(
    MODEL,
    name="narrative-report",
    output_type=PromptedOutput(NarrativeReport),
    retries=4,
    model_settings={"temperature": 0.3},
    instructions=(
        "You are the Narrative Report Writer agent for the NoiPA dataset quality pipeline. "
        "NoiPA is the digital platform of the Italian Ministry of Economy and Finance (MEF) that manages salaries, "
        "timesheets, and tax/social-security obligations for employees of the Italian Public Administration.\n\n"

        "You receive a structured quality briefing derived from the pipeline's validation, remediation, "
        "cleaning, and verification stages. Produce an EXHAUSTIVE, professional, human-readable quality "
        "report in ENGLISH that a data steward, project manager, or auditor can use as a definitive "
        "reference document.\n\n"

        "MANDATORY SECTIONS (use these exact English headings):\n\n"

        "1. 'Dataset Overview' — Dataset name, total rows, total columns (original and after cleaning). "
        "Overall quality posture: total findings, actions applied, actions deferred to manual review. "
        "State the cleaned output file path. ALWAYS wrap any filesystem path in backticks (inline code) so "
        "Markdown does not eat backslashes on Windows paths — e.g. `C:\\Users\\...\\cleaned.csv`.\n\n"

        "2. 'Schema Validation' — Two subsections:\n"
        "   a) 'Column Renames': list EVERY column rename as a markdown table with columns: Original Name | New Name | Reason.\n"
        "   b) 'Type Casts': list EVERY dtype cast as a markdown table with columns: Column | Assigned Type | % Non-Null. "
        "   The % Non-Null value MUST come from the NON-NULL COUNTS block in the briefing — it already provides each column's "
        "   exact percentage, computed from the cleaned CSV. Copy that percentage verbatim. "
        "   Never estimate, interpolate, or invent these numbers. If a column cast does not appear in the NON-NULL COUNTS "
        "   block (e.g. the cast was not_needed and the column does not exist in the cleaned frame), leave the cell as 'n/a'. "
        "   Mention any casts that were planned but marked not_needed and why.\n\n"

        "3. 'Completeness Analysis' — For EACH column where placeholders were replaced, state: "
        "the column name, how many placeholder values were found, which placeholder tokens were detected "
        "(list the actual examples like 'n.d.', '-', '//', '?', 'unknown', empty string), "
        "and that they were replaced with null. Use a markdown table. "
        "Source of truth: the COMPLETENESS DETAILS block in the briefing — copy each column's "
        "missing_like_count, completeness %, and tokens verbatim. Do not invent placeholder examples. "
        "Also mention columns where placeholder replacement was planned but not needed.\n\n"

        "4. 'Format Consistency' — THIS IS WHERE THE MAJOR INCONSISTENCIES WERE FOUND AND FIXED. "
        "Begin with a one-sentence summary of how many columns had format issues. "
        "Then for EACH column where a cleaner was generated, use this EXACT structured format:\n"
        "### column_name\n"
        "- **Expected Pattern:** …\n"
        "- **Inconsistent Rows:** N\n"
        "- **Examples of bad values:** 'val1', 'val2', 'val3'\n"
        "- **Transformation applied:** …\n"
        "- **Clean example:** 'bad_value' → 'clean_value'\n"
        "- **Outcome:** Fixed / Mostly Fixed / Unchanged / Regression\n\n"
        "Use 'Fixed' when fully resolved, 'Mostly Fixed' when substantially improved with residual issues, "
        "'Unchanged' when no improvement, 'Regression' when worse. "
        "Never collapse multiple columns into a single paragraph — each gets its own ### heading and bullet list.\n\n"

        "MANDATORY RULE FOR CLEAN EXAMPLES: "
        "The 'Clean example' line MUST be copied verbatim from the CLEANER EXAMPLE TRANSFORMATIONS block in the briefing. "
        "Find the section for that exact column and quote ONE entry — both the original and the cleaned value, character for character, "
        "including any trailing '.000', time suffixes, leading zeros, or punctuation. "
        "Do NOT reformat the cleaned value. Do NOT trim it to 'look like ISO 8601' or 'look like YYYYMM'. "
        "Do NOT invent pairings when the transformation list does not contain a given input (for example, 'Rata 2024' does not imply any specific month — if the ground-truth list does not contain 'Rata 2024' with a cleaned month attached, pick a different input that IS in the list). "
        "If a column has no entries in the CLEANER EXAMPLE TRANSFORMATIONS block, OMIT the 'Clean example' bullet entirely for that column rather than fabricate one. "
        "Similarly, 'Examples of bad values' should be drawn from the BRIEFING's inconsistent-examples list, not invented.\n\n"

        "5. 'Anomaly Detection' — Cover numeric outliers and rare categories separately, grounded in the "
        "ANOMALY FINDINGS block in the briefing. Every column name, severity, affected_rows count, and example "
        "value MUST be quoted from that block. The 'evidence' line in each finding contains the IQR band or "
        "frequency threshold — quote it directly; do NOT compute or invent one. "
        "If the ANOMALY FINDINGS block is empty or absent, state clearly that no anomalies were detected and "
        "skip the rest of this section rather than fabricate entries. "
        "Explain why each flagged item was routed to manual review rather than auto-corrected, citing the "
        "'suggested_action' line from the block.\n\n"

        "6. 'Cross-Column Checks' — Every fact in this section MUST come from the CROSS-COLUMN FINDINGS "
        "block in the briefing. Each finding there has a check_type that maps to one subsection below — "
        "never list the same pair in more than one subsection, and never invent pairs that are not in the block.\n"
        "   a) Exact duplicate columns (check_type=exact_duplicate_columns). ONLY list a pair here if BOTH: "
        "      (i) a finding with check_type=exact_duplicate_columns exists in the block, AND "
        "      (ii) an action of type `drop_exact_duplicate_column` targeting that pair appears in APPLIED ACTIONS. "
        "      If either is missing, state 'No exact duplicate columns were detected.' "
        "      Do NOT claim a drop based on similarity % alone — 99.x% is a near-duplicate, not an exact duplicate.\n"
        "   b) Near-duplicate columns (check_type=near_duplicate_columns): list each pair with its similarity_pct "
        "      and affected_rows count exactly as given in the block. These pairs were NOT dropped; they require manual review.\n"
        "   c) Semantic conflicts (check_type=duplicate_semantic_conflict): list columns and affected_rows from the block.\n"
        "   d) Period/date consistency (check_type=year_month_period_mismatch or date_order_violation): "
        "      list the columns and affected_rows from the block.\n"
        "If the CROSS-COLUMN FINDINGS block is empty or absent, state 'No cross-column anomalies were detected.' "
        "and omit subsections (a)-(d).\n\n"

        "7. 'Row Duplicate Analysis' — Ground every fact in the DUPLICATE ROW GROUPS block in the briefing. "
        "State how many exact duplicate groups vs. near-duplicate groups exist, quoting the block's group counts "
        "and total affected rows. Give 2-3 concrete row-index examples, copying indices verbatim from the "
        "sample_indices lines (never invent indices). Quote the key_columns list from the block. "
        "Explain why exact duplicates were NOT auto-removed (conservative policy) and why near-duplicates require "
        "manual review, citing the 'evidence' line. "
        "If the block is empty or absent, state that no duplicate rows were detected and skip the rest of the section.\n\n"

        "8. 'Remediation Action Summary' — Provide a complete accounting:\n"
        "   a) Summary counts, in this exact order: Applied, Deferred, Failed, Not Needed. "
        "      These four buckets partition the action set and must sum to the total. "
        "      'Deferred' = proposed_not_applied (items routed to manual review). "
        "      'Not Needed' = actions that turned out to be redundant or already satisfied — it is a DIFFERENT category "
        "      from Deferred; do NOT merge the two into one count.\n"
        "   b) Breakdown by action type (markdown table): Action Type | Applied | Deferred | Failed | Not Needed.\n"
        "   c) Briefly explain the auto-apply policy: what is safe to auto-apply and what requires human review.\n\n"

        "9. 'Verification Outcome' — This section reports which format inconsistencies were cleaned and to what degree. "
        "Report the verification results: how many consistency findings were resolved, mostly fixed, unchanged, or regressed. "
        "Use the VERIFICATION DIFFS rows from the briefing verbatim. "
        "When a column was renamed (new_name differs from the original), list it under its NEW name in the "
        "cleaned CSV and note the rename arrow inline (e.g. 'rata (renamed from RATA): Fixed'). "
        "Do NOT list the pre-rename name on its own — it does not exist in the cleaned output. "
        "List each column with its Before status and After status using the labels: "
        "Fixed / Mostly Fixed / Unchanged / Regression. "
        "Explain what 'Fixed' means (inconsistent rows now conform to the expected pattern).\n\n"

        "10. 'Residual Risks & Manual Review Queue' — List ALL items requiring manual review: "
        "near-duplicate columns, semantic conflicts, near-duplicate rows, numeric outliers, proposed-not-applied actions. "
        "For each, state what action is needed and why it was not automated. "
        "If there are no unresolved risks from cleaning, state that explicitly.\n\n"

        "STYLE RULES:\n"
        "- Write in clear, professional English suitable for institutional documentation.\n"
        "- Every claim must cite concrete data from the briefing: column names, row counts, percentages, example values.\n"
        "- Use markdown tables for structured data (renames, casts, placeholders, near-duplicate pairs).\n"
        "- Use bullet lists for enumerated findings.\n"
        "- Each section body must be at least 150 words — this is an exhaustive report, not a summary.\n"
        "- The executive_summary must be 8-12 sentences and readable standalone, in English.\n"
        "- Do not invent facts not present in the input document.\n"
        "- Do not use generic filler phrases — every sentence must carry information.\n"
        "- recommendations must contain at least 5 actionable, prioritized items in English.\n\n"

        "OUTPUT: valid JSON matching NarrativeReport exactly. No raw markdown outside the JSON fields."
    ),
)


### 5.11 `narrative_frontmatter_agent`

Role: writes the title, executive summary, and recommendations for the final report.


In [ ]:
narrative_frontmatter_agent = Agent(
    MODEL,
    name="narrative-frontmatter",
    output_type=PromptedOutput(NarrativeFrontMatter),
    retries=4,
    model_settings={"temperature": 0.2},
    instructions=(
        "You write the front matter for the final quality report. "
        "Use only the attached briefing. Return valid JSON matching NarrativeFrontMatter exactly.\n\n"
        "RULES:\n"
        "- title must include the dataset name.\n"
        "- executive_summary must be 8-12 sentences in professional English.\n"
        "- recommendations must contain at least 3 concrete, prioritized actions in English.\n"
        "- Do not use backticks for ordinary labels, column names, percentages, or example values.\n"
        "- Do not invent facts not present in the briefing.\n"
        "- No markdown outside the JSON fields."
    ),
)


### 5.12 `narrative_section_agent`

Role: writes one report section at a time so the final narrative is less brittle than a single giant JSON output.


In [ ]:
narrative_section_agent = Agent(
    MODEL,
    name="narrative-section",
    output_type=PromptedOutput(NarrativeReportSection),
    retries=4,
    model_settings={"temperature": 0.2},
    instructions=(
        "You write exactly one section body for the final dataset quality report. "
        "Use only the attached section briefing. Return valid JSON matching NarrativeReportSection exactly.\n\n"
        "RULES:\n"
        "- heading must exactly match the requested section heading.\n"
        "- body must be markdown-formatted prose in professional English.\n"
        "- body must be at least 150 words and grounded in the provided facts only.\n"
        "- Use tables or bullet lists when they help clarity, but keep everything inside the body field.\n"
        "- Do not use backticks for ordinary column names, labels, values, row counts, or percentages.\n"
        "- Round percentages to one decimal place unless the briefing explicitly requires a different precision.\n"
        "- Do not invent facts, counts, examples, or file paths.\n"
        "- No markdown outside the JSON fields."
    ),
)


## 6. Production function definitions

This section shows the main orchestration entrypoints as real code cells. 
The full validation modules are shown immediately afterwards, but these entrypoints are useful because they are the thinnest readable map of the end-to-end pipeline.


### 6.1 Validation and bundling entrypoints

Role: these are the top-level functions that execute the validation half. They orchestrate profiling, agent calls, caching, and final bundling.


- `run_dtype_inference`: a thin entry-point called by `run_schema_validation`. It prepares the dtype-inference text attachment (column name + sampled values) and dispatches the one LLM call to the `dtype_inference_agent`. The returned `DatasetDtypeInference` drives all downstream dtype-aware decisions (completeness, consistency, remediation, cleaning).
- `run_schema_validation`: the main entry-point for the validation half. It runs deterministic schema profiling to build the schema handoff, calls the `schema_summary_agent` to get the narrative summary, runs all other agents to get structured findings, caches everything, and bundles it into a final `ValidationBundle` for downstream use by cleaning and reporting stages.

This will inform the schema profile and the validation analysis. The dtype inference agent looks at column samples and returns a mapping of column names to inferred pandas dtypes.

This will help the agent understand which columns are likely duplicates based on their names, even if they have different original names. For example, "RATA" and "rata" would both normalize to "rata" and be grouped together as potential duplicates.

Inner helpers (all from tools/ + cache/):
- load_dataset_frame: thin pd.read_csv wrapper
- build_dataset_profile: per-column dtype / null stats / samples
- run_dtype_inference: calls dtype_inference_agent on the profile
- normalized_schema_name / is_valid_schema_name / suggest_schema_name: deterministic naming rules
- build_schema_issues: naming + duplicate-column heuristics
- schema_summary_agent: writes the handoff summary
- load_schema_handoff / save_schema_handoff: JSON cache under Data/.validation_cache/

The handoff object will be passed to the next agent for summarization and then saved to disk for downstream use.

In [6]:
def run_dtype_inference(path: Path) -> DatasetDtypeInference:
    """Run the dtype inference agent on the given dataset path and return the inferred dtypes for each column."""
    df = load_dataset_frame(path)
    # Build the text document for the agent: include column names and samples for each column, formatted clearly.
    text = build_dtype_inference_text(df)
    # Attach the text as a document and run the agent with backoff retries. The agent will return a DatasetDtypeInference object.
    prompt = ["Infer the correct pandas dtype for each column based on the attached CSV sample.",attach_text_document(text)]
    print(f"[orchestrator][schema][dtype-inference] dataset='{path.stem}'", file=sys.stderr, flush=True)
    # Run the agent with backoff retries and return the output field, which contains the inferred dtypes.
    result = run_agent_with_backoff(dtype_inference_agent, prompt)
    return result.output

def run_schema_validation(path: Path, reuse_cache: bool = False) -> SchemaHandoff:
    """Run the schema validation agent on the given dataset path and return a SchemaHandoff object containing the analysis results."""
    if reuse_cache:
        # If reuse_cache is True, load the existing handoff from disk instead of re-running the analysis. This allows us to skip expensive agent calls when we just want to inspect or use the results.
        return load_schema_handoff(path)

    df = load_dataset_frame(path)

    # Run dtype inference first to get the agent's best guess at each column's dtype.
    dtype_inference = run_dtype_inference(path)
    # Build a mapping from column name to inferred dtype column for easy lookup.
    dtype_map = {col.column_name: col for col in dtype_inference.columns}
    dtype_overrides = {name: col.pandas_dtype for name, col in dtype_map.items()}

    # Build the dataset profile, which includes statistics and samples for each column. Pass the dtype overrides to ensure the profile uses the agent's inferred dtypes.
    profile = build_dataset_profile(df, path.stem, dtype_overrides=dtype_overrides)

    # Store columns with the same normalized name together to detect potential duplicates.
    duplicate_groups_by_name: dict[str, list[str]] = {}
    for col_name in df.columns:
        canonical = normalized_schema_name(col_name)
        duplicate_groups_by_name.setdefault(canonical, []).append(col_name)
    
    # Only keep groups with more than one column, as singletons are not duplicates.
    duplicate_groups = [SchemaDuplicateGroup(canonical_name=cn, columns=cols) for cn, cols in duplicate_groups_by_name.items() if len(cols) > 1]

    # Build the list of SchemaColumnEntry objects that contain the analysis for each column
    columns: list[SchemaColumnEntry] = []

    # Iterate over the column profiles from the dataset profile 
    for col_profile in profile.columns_profiles:
        name = col_profile.column_name
        dtype_col = dtype_map.get(name)
        # Determine the final pandas dtype and roles for this column, based on the agent's inference and the column profile.
        dtype_inference_choice = _normalize_dtype_inference_choice(name, dtype_col, col_profile)
        columns.append(SchemaColumnEntry(
            name = col_profile.column_name,
            pandas_dtype = dtype_inference_choice[0], # The final pandas dtype to assign to this column, based on the agent's inference and the profile statistics.
            numeric_role = dtype_inference_choice[1], # The numeric role assigned to this column based on the agent's inference and the profile statistics.
            string_role = dtype_inference_choice[2], # The string role assigned to this column based on the agent's inference and the profile statistics.
            detected_pattern = dtype_inference_choice[3], # The pattern detected for this column based on the agent's inference and the profile statistics.
            rationale = dtype_inference_choice[4], # The rationale for the dtype inference choice.
            non_null_rows = col_profile.non_null_rows,
            distinct_non_null_values = col_profile.distinct_non_null_values,
            numeric_parse_pct=col_profile.numeric_parse_pct,
            datetime_parse_pct=col_profile.datetime_parse_pct,
            empty_like_pct=col_profile.empty_like_pct,
            sample_values=col_profile.sample_values,
            naming_valid = is_valid_schema_name(name), # Check if the column name is valid according to the naming rules.
            naming_reason = naming_rule_reason(name) if not is_valid_schema_name(name) else None, # If the name is not valid, provide a reason why it violates the naming rules
            rename_suggestion = suggest_schema_name(name) if not is_valid_schema_name(name) else None,# If the name is not valid, suggest a valid name based on the original.
        ))

    # Build the list of schema issues based on the column analysis and duplicate groups. 
    issues = build_schema_issues(columns, duplicate_groups)

    # Create the SchemaHandoff object that contains the dataset name, column analysis, and detected issues.
    handoff = SchemaHandoff(
        dataset_name=path.stem,
        total_rows=len(df),
        total_columns=len(df.columns),
        columns=columns,
        issues=issues,
        duplicate_groups=duplicate_groups,
    )
    print(f"[orchestrator][schema][summary] dataset='{path.stem}'", file=sys.stderr, flush=True)

    # Run the schema summary agent to generate a concise summary of the schema analysis, which will be included in the handoff for downstream use.
    prompt = f"Summarize the provided schema analysis for dataset {path.stem}. Do not infer new findings."                                                                                                                                                  
    result = run_agent_with_backoff(schema_summary_agent, [prompt, attach_profile_text(handoff)])
    
    # Update the handoff with the generated summary and save it to disk for downstream stages to consume.
    handoff = handoff.model_copy(update={"summary": result.output.summary})
    save_schema_handoff(path, handoff)
    return handoff

### 6.1.2 `run_completeness_analysis`

Role: top-level completeness stage. It computes deterministic missingness facts, then asks the completeness agent to format them into a structured report.
run_completeness_analysis: missingness + placeholder detection (1 LLM call).
Inner helpers: 
- build_completeness_profile:  missing-like counts, placeholder tokens, per-column rates
- attach_profile_text: sends the profile to the agent as a text document
- run_agent_with_backoff: robust wrapper around pydantic-ai calls
- load_completeness / save_completeness: JSON cache

In [ ]:
def run_completeness_analysis(path: Path, reuse_cache: bool = False) -> CompletenessAnalysisReport:
    """Build a completeness profile for the dataset and return the agent's structured analysis.
    The profile is computed locally from the raw data, then handed to the agent which
    identifies missing values, placeholder tokens, and sparse columns. The result is cached.
    """
    if reuse_cache:
        return load_completeness(path)
    df = load_dataset_frame(path)
    # Build a per-column completeness profile to attach to the agent prompt
    profile = build_completeness_profile(df, path.stem)
    prompt = [
        (
            f"Analyze the attached completeness profile for dataset {path.stem}. "
            "Use Python in code execution to inspect the profile document. "
            "Use the provided metrics to summarize per-column completeness, detect missing-like and placeholder values, "
            "identify actual placeholder tokens present in the dataset, and flag sparse columns that may be candidates for removal or investigation."
        ),
        attach_profile_text(profile),
    ]
    print(f"[orchestrator][completeness] dataset='{path.stem}'", file=sys.stderr, flush=True)
    result = run_agent_with_backoff(completeness_analysis_agent, prompt)
    report = result.output
    save_completeness(path, report)
    return report

### 6.1.3 `run_format_consistency_validation`

Role: top-level consistency stage. It first tries a deterministic schema-guided fast path, then falls back to the format-consistency agent for ambiguous columns.
run_format_consistency_validation: per-column format-consistency check.
Fast path: deterministic schema-guided acceptance (no LLM).
Slow path: format_consistency_agent when shape / pattern is ambiguous.
Inner helpers:
- build_column_format_facts: dominant shape, sample outliers, semantic hint
- run_column_format_check: fast-path acceptance + slow-path fallback
- load_consistency / save_consistency: JSON cache


In [ ]:
def run_format_consistency_validation(path: Path, reuse_cache: bool = False, read_as_str: bool = False, max_workers: int = 1) -> ConsistencyValidationReport:
    """Run format consistency checks for all columns and return the combined report.

    When max_workers > 1, columns are checked concurrently using async tasks. Schema context
    is loaded if available to enable the fast path for columns with known patterns.
    """
    if max_workers < 1:
        raise ValueError("max_workers must be at least 1.")
    if reuse_cache:
        return load_consistency(path)
    df = load_dataset_frame(path, dtype=str if read_as_str else None)
    format_findings: list[FormatConsistencyFinding] = []

    # Load schema to provide pattern and dtype context to each column check
    handoff = load_schema_handoff(path)
    schema_map = {col.name: col for col in handoff.columns}

    column_names = list(df.columns)
    # Use the synchronous path for single-worker runs to keep the event loop simple
    if max_workers == 1 or len(column_names) <= 1:
        reports = []
        for column_name in column_names:
            schema_entry = schema_map.get(column_name)
            # Compute the report for this column and append it to the list of reports
            reports.append(run_column_format_check(df, column_name, path.stem, "original validation", schema_entry=schema_entry))
    else:
        # Cap the worker count so we never spawn more tasks than there are columns
        worker_count = min(max_workers, len(column_names))
        print(
            f"[orchestrator][consistency] running {len(column_names)} column checks with {worker_count} async workers",
            file=sys.stderr,
            flush=True,
        )
        # Run all column checks concurrently and collect the reports as they complete, then reorder them to match the original column order
        reports = asyncio.run(_run_column_format_checks_async(df, column_names, path.stem, schema_map, worker_count))

    # Collect only the columns that actually have a finding
    for result in reports:
        if result.finding is not None:
            format_findings.append(result.finding)

    # Build the final consistency report with a summary of the findings
    report = ConsistencyValidationReport(
        dataset_name=path.stem,
        total_rows=len(df),
        format_consistency_findings=format_findings,
        summary=(
            f"Analyzed all {len(df.columns)} columns individually for format consistency and "
            f"detected {len(format_findings)} format issues."
        ),
    )
    # Save it to disk for downstream stages to consume.
    save_consistency(path, report)
    return report

### 6.1.4 `run_anomaly_detection`

Role: top-level anomaly stage. It uses deterministic heuristics for numeric outliers and rare categories, then asks a summary agent to write the short handoff summary.
run_anomaly_detection: numeric outliers + rare categories (heuristics + 1 summary LLM call).
Inner helpers:
- detect_numeric_outlier_candidates: IQR-style numeric outliers
- detect_rare_category_candidates: rare-value heuristic for low-cardinality strings
- anomaly_summary_agent: compact summary of findings
- load_anomaly / save_anomaly: JSON cache

In [ ]:
def run_anomaly_detection(path: Path, reuse_cache: bool = False) -> AnomalyDetectionReport:
    if reuse_cache:
        return load_anomaly(path)

    df = load_dataset_frame(path)
    try:
        handoff = load_schema_handoff(path)
        schema_columns = handoff.columns
        suppressed_columns = _duplicate_semantic_suppressed_columns(handoff.columns, handoff.duplicate_groups)
    except FileNotFoundError:
        schema_columns = []
        suppressed_columns = set()

    findings = [
        AnomalyFinding(**finding)
        for finding in (
            detect_numeric_outlier_candidates(df, schema_columns)
            + detect_rare_category_candidates(df, schema_columns)
        )
        if finding["column_name"] not in suppressed_columns
    ]
    findings.sort(key=lambda finding: (-finding.affected_rows, finding.column_name, finding.anomaly_type))
    fallback_summary = (
        f"Detected {len(findings)} anomaly findings across numeric outliers and rare categorical values."
        if findings
        else "No anomaly findings were detected by the current heuristic checks."
    )
    report = AnomalyDetectionReport(
        dataset_name=path.stem,
        total_rows=len(df),
        total_columns=len(df.columns),
        findings=findings,
        summary=fallback_summary,
    )
    print(f"[orchestrator][anomaly][summary] dataset='{path.stem}'", file=sys.stderr, flush=True)
    report = report.model_copy(
        update={
            "summary": summarize_validation_report(
                anomaly_summary_agent,
                (
                    f"Summarize the provided anomaly-detection findings for dataset {path.stem}. "
                    "Do not infer new findings or alter the provided findings."
                ),
                report,
                fallback_summary,
            )
        }
    )
    save_anomaly(path, report)
    return report


### 6.1.5 `run_cross_column_validation`

Role: top-level cross-column stage. It looks for duplicate-like columns, semantic conflicts, and temporal mismatches before summarizing them for the handoff.


In [ ]:
# run_cross_column_validation: column-pair and relational checks (heuristics + 1 summary LLM call).
# Inner helpers:
# - detect_duplicate_like_columns           -> near-duplicate columns by value similarity
# - detect_duplicate_semantic_conflicts     -> same concept expressed as different columns
# - detect_year_month_period_mismatches     -> period / date coherence heuristics
# - detect_date_order_violations            -> start-date vs end-date ordering
# - cross_column_summary_agent              -> summary of findings
# - load_cross_column / save_cross_column   -> JSON cache

def run_cross_column_validation(path: Path, reuse_cache: bool = False) -> CrossColumnValidationReport:
    if reuse_cache:
        return load_cross_column(path)

    df = load_dataset_frame(path)
    try:
        handoff = load_schema_handoff(path)
        schema_columns = handoff.columns
        duplicate_groups = handoff.duplicate_groups
    except FileNotFoundError:
        schema_columns = []
        duplicate_groups = []

    findings = [
        CrossColumnFinding(**finding)
        for finding in (
            detect_duplicate_like_columns(df, schema_columns)
            + detect_duplicate_semantic_conflicts(df, duplicate_groups)
            + detect_year_month_period_mismatches(df, schema_columns)
            + detect_date_order_violations(df, schema_columns)
        )
    ]
    findings.sort(key=lambda finding: (-finding.affected_rows, ",".join(finding.columns), finding.check_type))
    fallback_summary = (
        f"Detected {len(findings)} cross-column consistency findings."
        if findings
        else "No cross-column consistency findings were detected by the current rule set."
    )
    report = CrossColumnValidationReport(
        dataset_name=path.stem,
        total_rows=len(df),
        findings=findings,
        summary=fallback_summary,
    )
    print(f"[orchestrator][cross-column][summary] dataset='{path.stem}'", file=sys.stderr, flush=True)
    report = report.model_copy(
        update={
            "summary": summarize_validation_report(
                cross_column_summary_agent,
                (
                    f"Summarize the provided cross-column validation findings for dataset {path.stem}. "
                    "Do not infer new findings or alter the provided findings."
                ),
                report,
                fallback_summary,
            )
        }
    )
    save_cross_column(path, report)
    return report


### 6.1.6 `run_duplicate_detection`

Role: top-level duplicate stage. It finds exact duplicate rows and near-duplicate groups, then summarizes them for downstream review.


In [ ]:
# run_duplicate_detection: exact + near-duplicate row groups (heuristics + 1 summary LLM call).
# Inner helpers:
# - infer_duplicate_key_columns             -> chooses the keys for grouping
# - detect_exact_duplicate_groups           -> deterministic exact-row detection
# - detect_near_duplicate_groups            -> heuristic near-duplicate grouping
# - duplicate_summary_agent                 -> summary of findings
# - load_duplicates / save_duplicates       -> JSON cache

def run_duplicate_detection(path: Path, reuse_cache: bool = False) -> DuplicateDetectionReport:
    if reuse_cache:
        return load_duplicates(path)

    df = load_dataset_frame(path)
    try:
        handoff = load_schema_handoff(path)
        schema_columns = handoff.columns
    except FileNotFoundError:
        schema_columns = []

    key_columns = infer_duplicate_key_columns(schema_columns, df)
    groups = [
        DuplicateRecordGroup(**group)
        for group in (
            detect_exact_duplicate_groups(df)
            + detect_near_duplicate_groups(df, key_columns)
        )
    ]
    fallback_summary = (
        f"Detected {len(groups)} duplicate-record groups."
        if groups
        else "No duplicate-record groups were detected by the current exact and near-duplicate checks."
    )
    report = DuplicateDetectionReport(
        dataset_name=path.stem,
        total_rows=len(df),
        groups=groups,
        summary=fallback_summary,
    )
    print(f"[orchestrator][duplicates][summary] dataset='{path.stem}'", file=sys.stderr, flush=True)
    report = report.model_copy(
        update={
            "summary": summarize_validation_report(
                duplicate_summary_agent,
                (
                    f"Summarize the provided duplicate-detection findings for dataset {path.stem}. "
                    "Do not infer new findings or alter the provided findings."
                ),
                report,
                fallback_summary,
            )
        }
    )
    save_duplicates(path, report)
    return report


### 6.1.7 `build_validation_results`

Role: orchestration entrypoint for the whole validation half. It runs or reuses all six validation stages and bundles them into a single `OrchestrationStepResult`.


In [ ]:
# build_validation_results: one-shot orchestrator for the validation half (no LLM of its own).
# It simply chains all six run_* stages above, honoring the reuse_* flags for cached artifacts.
# Inner helpers:
# - save_validation_results                 -> persists the bundled OrchestrationStepResult

def build_validation_results(
    path: Path,
    reuse_schema: bool = False,
    reuse_completeness: bool = False,
    reuse_consistency: bool = False,
) -> OrchestrationStepResult:
    schema_validation = run_schema_validation(path, reuse_cache=reuse_schema)
    completeness_analysis = run_completeness_analysis(path, reuse_cache=reuse_completeness)
    consistency_validation = run_format_consistency_validation(path, reuse_cache=reuse_consistency)
    print(f"[orchestrator][anomaly] dataset='{path.stem}'", file=sys.stderr, flush=True)
    anomaly_detection = run_anomaly_detection(path)
    print(f"[orchestrator][cross-column] dataset='{path.stem}'", file=sys.stderr, flush=True)
    cross_column_validation = run_cross_column_validation(path)
    print(f"[orchestrator][duplicates] dataset='{path.stem}'", file=sys.stderr, flush=True)
    duplicate_detection = run_duplicate_detection(path)
    validation_results = OrchestrationStepResult(
        schema_validation=schema_validation,
        completeness_analysis=completeness_analysis,
        consistency_validation=consistency_validation,
        anomaly_detection=anomaly_detection,
        cross_column_validation=cross_column_validation,
        duplicate_detection=duplicate_detection,
    )
    save_validation_results(path, validation_results)
    return validation_results


## 6.2 Host-side validator

Between the validation half and the cleaning half sits a pure-Python guard with no LLM calls: `validate_generated_cleaner_program`. This is the function that makes the generator / critic loop actually converge.

Whenever the generator agent produces a `ColumnCleanerProgram`, this validator loads the generated function in a sandboxed namespace and exercises it against the dominant and inconsistent example values from the request. It emits one `CleanerValidationIssue` per detected problem, covering the failure categories defined by the `VALIDATION_FAILURE_CATEGORY` Literal.

The issues are fed back to the `cleaner_repair_critic_agent`, which produces a structured repair brief that drives the next generator attempt.

### 6.2.1 Categories produced by the validator

| Category | Meaning |
|---|---|
| `program_mismatch` | Program declares a different column name than the request |
| `non_self_contained_function` | Cleaner references outer-scope names (NameError on load or call) |
| `runtime_exception` | Cleaner raised an exception other than NameError |
| `shadowed_specific_branch` | Generic `if '<sep>' in s:` above a more specific branch on the same separator |
| `dominant_value_modified` | Cleaner rewrote an already-valid dominant example (identity violation) |
| `outlier_unchanged` | Cleaner returned an inconsistent example unchanged |
| `wrong_output_shape` | Output did not match the dominant-example structural shape |
| `not_parseable_as_target_dtype` | Output could not be parsed as `Int64` / `Float64` / `datetime64[ns]` / `boolean` |
| `not_matching_target_pattern` | Output did not match the target numeric pattern (code vs. measure) |


### 6.2.2 `validate_generated_cleaner_program` (the function itself)

Copied verbatim from `cleaning/validation.py`. Every failure category above originates in one of the branches of this function.

In [ ]:
# validate_generated_cleaner_program: pure-Python, no LLM call.
# Helpers (defined in cleaning/validation.py, imported here by name only):
# - load_cleaner_callable                   -> loads program.python_code into a callable in an isolated namespace
# - detect_shadowed_delimiter_branches      -> static check for unreachable elif branches
# - dominant_output_shape                   -> mode of the value_shape() of dominant examples
# - requires_fixed_output_shape             -> True unless target_dtype is numeric
# - dominant_datetime_example               -> canonical datetime example for shape matching
# - matches_dominant_datetime_format        -> regex-from-example datetime layout check
# - is_parseable_output                     -> tries pd.to_datetime / pd.to_numeric on the output
# - matches_request_target_pattern          -> numeric code vs. measure pattern check
# - build_validation_issue / build_runtime_exception_issue -> issue constructors
# - _suggest_corrected_datetime / _diagnose_datetime_component_order
#                                           -> datetime-specific repair hints shown to the critic

from cleaning.validation import (
    load_cleaner_callable,
    detect_shadowed_delimiter_branches,
    dominant_output_shape,
    requires_fixed_output_shape,
    dominant_datetime_example,
    matches_dominant_datetime_format,
    is_parseable_output,
    matches_request_target_pattern,
    build_validation_issue,
    build_runtime_exception_issue,
    _suggest_corrected_datetime,
    _diagnose_datetime_component_order,
    value_shape,
)

def validate_generated_cleaner_program(
    request: ColumnCleaningRequest,
    program: ColumnCleanerProgram,
) -> list[CleanerValidationIssue]:
    issues: list[CleanerValidationIssue] = []

    # 1. Name must match the request.
    if program.column_name != request.column_name:
        issues.append(
            build_validation_issue(
                category="program_mismatch",
                severity="high",
                message=f"Program column_name was {program.column_name!r}, expected {request.column_name!r}.",
                expected_behavior=f"column_name must equal {request.column_name!r}.",
            )
        )

    # 2. Must load into a callable (sandboxed; no scope leaks).
    try:
        cleaner = load_cleaner_callable(program)
    except NameError as error:
        return [build_validation_issue(
            category="non_self_contained_function", severity="high",
            message=f"Generated cleaner referenced an undefined name: {error}.",
            expected_behavior="the function must be fully self-contained.",
        )]
    except Exception as error:
        return [build_validation_issue(
            category="runtime_exception", severity="high",
            message=f"Generated cleaner could not be loaded: {error}",
            expected_behavior="python_code must load without exceptions.",
        )]

    # 3. Static check for unreachable branches.
    issues.extend(detect_shadowed_delimiter_branches(program))

    target_shape = dominant_output_shape(request)
    require_fixed_shape = requires_fixed_output_shape(request)
    target_datetime_example = dominant_datetime_example(request)

    # 4. Dominants must be returned unchanged (identity).
    for value in request.dominant_example_values:
        try:
            cleaned = cleaner(value)
        except Exception as error:
            issues.append(build_runtime_exception_issue(
                stage_label="Dominant example", input_value=value, error=error))
            continue
        if cleaned != value:
            issues.append(build_validation_issue(
                category="dominant_value_modified", severity="high",
                message=f"Dominant example {value!r} changed to {cleaned!r}; already-valid values must be preserved exactly.",
                input_value=value, actual_output=None if cleaned is None else str(cleaned),
                expected_behavior="return the dominant example unchanged.",
            ))

    # 5. Outliers: must be transformed, match target shape/dtype/pattern.
    for value in request.example_inconsistent_values:
        try:
            cleaned = cleaner(value)
        except Exception as error:
            issues.append(build_runtime_exception_issue(
                stage_label="Inconsistent example", input_value=value, error=error))
            continue
        if cleaned is None:
            continue  # cleaner chose to null the outlier, acceptable
        cleaned_str = str(cleaned)
        if cleaned == value:
            issues.append(build_validation_issue(
                category="outlier_unchanged", severity="high",
                message=f"Inconsistent example {value!r} was returned unchanged.",
                input_value=value, actual_output=cleaned_str,
                expected_behavior="the outlier should be normalized, not passed through unchanged.",
            ))

        # 5a. Shape: non-numeric targets must keep the dominant structural shape.
        if require_fixed_shape:
            if request.target_dtype == "datetime64[ns]" and target_datetime_example is not None:
                if matches_dominant_datetime_format(cleaned_str, request) is False:
                    suggested = _suggest_corrected_datetime(value, target_datetime_example)
                    diagnosis = _diagnose_datetime_component_order(value, cleaned_str, target_datetime_example)
                    issues.append(build_validation_issue(
                        category="wrong_output_shape", severity="medium",
                        message=(f"Inconsistent example {value!r} cleaned to {cleaned_str!r}, which does not match "
                                 f"the canonical datetime format {target_datetime_example!r}.{diagnosis}"),
                        input_value=value, actual_output=cleaned_str,
                        expected_behavior=f"produce output matching {target_datetime_example!r}.",
                    ))
            elif target_shape is not None and value_shape(cleaned_str) != target_shape:
                issues.append(build_validation_issue(
                    category="wrong_output_shape", severity="medium",
                    message=(f"Inconsistent example {value!r} cleaned to {cleaned_str!r} with shape "
                             f"{value_shape(cleaned_str)!r}, expected {target_shape!r}."),
                    input_value=value, actual_output=cleaned_str,
                    expected_behavior=f"produce output matching shape {target_shape!r}.",
                ))

        # 5b. Must be parseable as the target dtype.
        if not is_parseable_output(cleaned_str, request.target_dtype):
            issues.append(build_validation_issue(
                category="not_parseable_as_target_dtype", severity="high",
                message=f"{value!r} cleaned to {cleaned_str!r}, not parseable as {request.target_dtype}.",
                input_value=value, actual_output=cleaned_str,
                expected_behavior=f"produce a value parseable as {request.target_dtype}.",
            ))
            continue

        # 5c. For numeric dtypes, check the target pattern (code vs. measure).
        if matches_request_target_pattern(cleaned_str, request) is False:
            issues.append(build_validation_issue(
                category="not_matching_target_pattern", severity="high",
                message=(f"{value!r} cleaned to {cleaned_str!r}, which does not match "
                         f"the target pattern {request.expected_pattern!r}."),
                input_value=value, actual_output=cleaned_str,
                expected_behavior=f"produce a value matching {request.expected_pattern!r}.",
            ))

    return issues

### 6.3 Cleaning, verification, and reporting entrypoints

Role: these are the orchestration functions for the cleaning half. Read them as a sequence: plan what to do, build cleaner requests, run the generator/validator/critic loop, apply the accepted cleaners, verify the result, and only then assemble the final report.


### 6.3.1 `run_remediation_planning`

Role: deterministic planning step that converts validation findings into a concrete list of actions, including renames, dtype casts, placeholder replacement, cleaner generation, and manual review items.


In [ ]:
# run_remediation_planning: deterministic. No LLM.
# Walks the validation bundle, emits a flat RemediationPlan of typed actions.
# Inner helpers:
# - _resolve_validation_results             -> loads cached bundle or runs validation
# - build_remediation_plan                  -> rules engine producing RemediationAction objects
# - load_remediation_plan / save_remediation_plan -> JSON cache

def run_remediation_planning(
    path: Path,
    validation_results: OrchestrationStepResult | None = None,
    reuse_saved_validation: bool = False,
    reuse_saved_remediation: bool = False,
) -> RemediationPlan:
    if reuse_saved_remediation:
        try:
            return load_remediation_plan(path)
        except FileNotFoundError:
            pass

    resolved_validation = _resolve_validation_results(path, validation_results, reuse_saved_validation)
    plan = build_remediation_plan(resolved_validation)
    save_remediation_plan(path, plan)
    return plan


### 6.3.2 `build_column_cleaning_request`

Role: merges one consistency finding, one column-format profile, and one schema entry into the exact prompt contract used by the cleaner generator.


In [ ]:
# build_column_cleaning_request: merges schema facts + format facts + consistency finding
# into the exact ColumnCleaningRequest the generator agent receives.
# Inner helpers:
# - _build_datetime_expected_pattern / _augment_datetime_strategy
#                                           -> datetime-specific prompt reinforcement

def build_column_cleaning_request(
    dataset_name: str,
    column_name: str,
    finding: Any,
    format_facts: Any,
    schema_entry: Any | None = None,
) -> ColumnCleaningRequest:
    example_inconsistent_values = list(dict.fromkeys(finding.example_inconsistent_values))
    if not example_inconsistent_values:
        example_inconsistent_values = list(
            dict.fromkeys(example.value for example in format_facts.inconsistent_examples)
        )

    target_dtype = schema_entry.pandas_dtype if schema_entry else None
    target_role = schema_entry.numeric_role or schema_entry.string_role if schema_entry else None
    expected_pattern = finding.expected_pattern
    suggested_strategy = finding.suggested_strategy

    if target_dtype == "datetime64[ns]":
        expected_pattern = _build_datetime_expected_pattern(format_facts, expected_pattern)
        suggested_strategy = _augment_datetime_strategy(format_facts, suggested_strategy)

    return ColumnCleaningRequest(
        dataset_name=dataset_name,
        column_name=column_name,
        expected_pattern=expected_pattern,
        semantic_hint=format_facts.semantic_hint,
        target_dtype=target_dtype,
        target_role=target_role,
        dominant_shape=format_facts.dominant_shape,
        dominant_example_values=format_facts.dominant_example_values,
        example_inconsistent_values=example_inconsistent_values,
        suggested_strategy=suggested_strategy,
    )


### 6.3.3 `run_column_cleaner_program`

Role: generator/critic loop for one column. It asks for a cleaner, validates it host-side, and if needed asks the critic for a repair brief before retrying.


#### Why this loop has three roles instead of one

This is the most important mechanism in the cleaning half, so it is worth stating the division of labor explicitly:

- The **generator** writes candidate Python code for one column.
- The **host-side validator** executes deterministic checks on that code without trusting the model's own self-assessment.
- The **critic** reads the validator's failures and writes a compact repair diagnosis for the next retry.

So the loop is not "generator versus discriminator" in the GAN sense. It is a **generator -> deterministic validator -> repair critic** pattern. The validator is the authority for correctness, while the critic is the agent that translates host-side failures into a better next prompt.


In [ ]:
# run_column_cleaner_program: the generator / critic / host-validator loop for one column.
# Per attempt:  build prompt -> generator agent -> host validator -> (on failure) critic -> retry.
# Inner helpers:
# - run_agent_with_backoff                  -> calls generator and critic agents
# - validate_generated_cleaner_program      -> host-side sandbox validator (defined in Sec. 6.2)
# - run_cleaner_repair_critic               -> turns host issues into a structured retry brief
# - _stagnation_temperature                 -> bumps temperature when repeated failures stagnate

def run_column_cleaner_program(
    dataset_name: str,
    request: ColumnCleaningRequest,
    max_attempts: int = 10,
    on_event: ProgressCallback | None = None,
) -> ColumnCleanerProgram:
    """Generator / critic / host-validator loop for a single column.

    Per attempt:
      1. Build the generator prompt (with failure context + optional stagnation brief).
      2. Ask the generator agent for a self-contained cleaner program.
      3. Validate the program host-side (no LLM).
      4. If valid, return the verified program.
      5. Otherwise: detect stagnation, invoke the critic, feed its diagnosis forward.
    """
    progress = _make_progress(on_event)
    previous_program: ColumnCleanerProgram | None = None
    validation_issues: list[CleanerValidationIssue] = []
    repair_diagnosis: CleanerRepairDiagnosis | None = None
    last_fingerprint: tuple[str, ...] | None = None
    consecutive_stagnant = 0

    for attempt in range(1, max_attempts + 1):
        stagnation = consecutive_stagnant >= 1
        progress.attempt_start(request.column_name, attempt, max_attempts, stagnation)

        prompt = _build_cleaner_generation_prompt(
            dataset_name, request,
            previous_program=previous_program,
            validation_issues=validation_issues,
            repair_diagnosis=repair_diagnosis,
            attempt_number=attempt,
            stagnation_detected=stagnation,
        )
        model_settings = {"temperature": _stagnation_temperature(consecutive_stagnant)} if stagnation else None
        try:
            program = run_agent_with_backoff(
                column_cleaner_generator_agent, prompt,
                usage_limits=GENERATOR_USAGE_LIMITS,
                model_settings=model_settings,
            ).output
        except UsageLimitExceeded as error:
            raise ValueError(
                "Generator exceeded the one-code-execution limit. "
                "This prevents hidden self-repair loops; simplify the prompt or raise the explicit limit if needed."
            ) from error

        validation_issues = validate_generated_cleaner_program(request, program)
        if not validation_issues:
            progress.generator_accepted(request.column_name, attempt)
            return rebuild_verified_program(request, program)
        progress.validation_failed(request.column_name, attempt, max_attempts, validation_issues[0])

        fingerprint = validation_issue_fingerprint(validation_issues)
        same_code = previous_program is not None and program.python_code.strip() == previous_program.python_code.strip()
        repeated_failure = last_fingerprint is not None and fingerprint == last_fingerprint
        if same_code or repeated_failure:
            reason = "repeated the same code" if same_code else "repeated the same host-side failures"
            progress.stagnation_noted(request.column_name, attempt, max_attempts, reason)
            consecutive_stagnant += 1
        else:
            consecutive_stagnant = 0
        previous_program, last_fingerprint = program, fingerprint

        if attempt < max_attempts:
            progress.critic_started(request.column_name, len(validation_issues))
            repair_diagnosis = run_cleaner_repair_critic(dataset_name, request, program, validation_issues)
            progress.critic_diagnosis(request.column_name, repair_diagnosis)
            if not repair_diagnosis.should_retry:
                failure_lines = "\n".join(f"- {format_validation_issue(issue)}" for issue in validation_issues[:10])
                raise ValueError(
                    f"Cleaner generation stopped for column '{request.column_name}' because the critic advised against retrying: "
                    f"{repair_diagnosis.root_cause}\n{failure_lines}"
                )

    failure_lines = "\n".join(f"- {format_validation_issue(issue)}" for issue in validation_issues[:10])
    raise ValueError(
        f"Cleaner generation failed local validation for column '{request.column_name}' after {max_attempts} attempts:\n"
        f"{failure_lines}"
    )


### 6.3.4 `run_cleaner_generation`

Role: dataset-level generation driver. It loops over all inconsistent columns, builds one request per column, and saves the accepted cleaners plus the manifest.


In [ ]:
# run_cleaner_generation: dataset-level driver that runs run_column_cleaner_program for every
# inconsistent column, optionally in parallel via a semaphore-controlled worker pool.
# Inner helpers:
# - _generation_lock                        -> prevents overlapping runs on the same dataset
# - _run_cleaner_generation_locked          -> inner concurrency-safe driver
# - run_format_consistency_validation       -> re-loads the per-column findings
# - _build_cleaning_requests                -> builds one ColumnCleaningRequest per finding

def run_cleaner_generation(
    path,
    reuse_consistency: bool = False,
    column_name: str | None = None,
    max_attempts: int = 10,
    max_workers: int = 1,
    on_event: ProgressCallback | None = None,
) -> list[GeneratedCleanerArtifact]:
    if max_attempts < 1:
        raise ValueError("max_attempts must be at least 1.")
    if max_workers < 1:
        raise ValueError("max_workers must be at least 1.")

    with _generation_lock(path):
        return _run_cleaner_generation_locked(path, reuse_consistency, column_name, max_attempts, max_workers, on_event)


### 6.3.5 `run_cleaner_application_with_plan`

Role: applies the deterministic remediation plan and the generated cleaners to the dataframe, then writes the cleaned CSV and cleaning summary.


In [ ]:
# run_cleaner_application_with_plan: executes the remediation plan against the raw CSV.
# Fixed safe order: rename columns -> replace placeholders with null -> apply generated cleaners
# -> drop exact-duplicate columns -> cast to target dtypes. Emits per-cleaner execution reports.
# Inner helpers:
# - _apply_column_renames / _apply_placeholder_nulls / _apply_cleaner_to_column / _cast_dtypes
# - _update_action_statuses                 -> marks each action applied / proposed / failed
# - save_cleaner_manifest                   -> writes the on-disk manifest of accepted cleaners

def run_cleaner_application_with_plan(
    path: Path,
    remediation_plan: RemediationPlan | None = None,
    on_event=None,
) -> tuple[CleaningReport, list[ColumnCleanerExecutionReport], RemediationPlan | None]:
    def _emit(message: str) -> None:
        if on_event is None:
            return
        try:
            on_event(message)
        except Exception:
            pass

    if remediation_plan is None:
        try:
            remediation_plan = load_remediation_plan(path)
        except FileNotFoundError:
            remediation_plan = None
    remediation_plan = _clone_remediation_plan(remediation_plan)
    actions = remediation_plan.actions if remediation_plan is not None else []

    try:
        artifacts = load_cleaner_manifest(path)
    except FileNotFoundError:
        artifacts = []
    df = load_dataset_frame(path)
    rows_before = len(df)
    columns_before = len(df.columns)
    execution_reports: list[ColumnCleanerExecutionReport] = []
    applied_artifacts: list[GeneratedCleanerArtifact] = []
    unresolved_risks: list[str] = []

    print(f"\n[apply] step 1 - format cleaners ({len(artifacts)} columns)", file=sys.stderr)
    _emit(f"step 1 — running {len(artifacts)} format cleaner{'s' if len(artifacts) != 1 else ''}")
    if not artifacts:
        print("  no cleaner manifest found or no generated cleaners recorded.", file=sys.stderr)
    for idx, artifact in enumerate(artifacts, start=1):
        program = _load_artifact_program(artifact)
        if program is None:
            unresolved_risks.append(f"{artifact.column_name}: cleaner file missing at {artifact.code_path}")
            continue
        if artifact.column_name not in df.columns:
            unresolved_risks.append(f"{artifact.column_name}: source column missing from dataset")
            continue

        cleaned_series, report = apply_cleaner_to_series(df[artifact.column_name], program)
        execution_reports.append(report)

        if report.execution_ok and cleaned_series is not None:
            df[artifact.column_name] = cleaned_series
            print(f"  '{artifact.column_name}' OK - {report.changed_rows} rows changed", file=sys.stderr)
            _emit(f"  [{idx}/{len(artifacts)}] ✓ '{artifact.column_name}' — {report.changed_rows} rows changed")
        else:
            print(f"  '{artifact.column_name}' FAILED - {'; '.join(report.unresolved_risks)}", file=sys.stderr)
            _emit(f"  [{idx}/{len(artifacts)}] ✗ '{artifact.column_name}' failed")
            unresolved_risks.extend(f"{artifact.column_name}: {risk}" for risk in report.unresolved_risks)

        applied_artifacts.append(
            GeneratedCleanerArtifact(
                column_name=artifact.column_name,
                function_name=artifact.function_name,
                code_path=artifact.code_path,
                changed_rows=report.changed_rows,
                summary=report.summary,
                example_transformations=list(artifact.example_transformations),
            )
        )

    execution_by_column = {report.column_name: report for report in execution_reports}
    for action in actions:
        if action.action_type != "generate_cleaner":
            continue
        report = execution_by_column.get(str(action.target.get("column_name", "")))
        if report is None:
            action.status = "failed"
        elif report.execution_ok:
            action.status = "applied"
        else:
            action.status = "failed"

    failed = [report for report in execution_reports if not report.execution_ok]
    print(
        f"\n  execution summary: {len(execution_reports) - len(failed)}/{len(execution_reports)} succeeded"
        + (f", {len(failed)} FAILED: {[report.column_name for report in failed]}" if failed else ""),
        file=sys.stderr,
    )

    print("\n[apply] step 2 - placeholder -> null (from completeness cache)", file=sys.stderr)
    _emit("step 2 — replacing placeholder tokens with null")
    df, total_replaced, placeholder_by_column = _apply_placeholder_nulls(df, path)
    print(f"  total placeholder replacements: {total_replaced}", file=sys.stderr)
    _emit(f"  {total_replaced} placeholder values replaced across {len(placeholder_by_column)} columns")
    for action in actions:
        if action.action_type != "replace_placeholders_with_null":
            continue
        column_name = str(action.target.get("column_name", ""))
        action.status = "applied" if placeholder_by_column.get(column_name, 0) > 0 else "not_needed"

    print("\n[apply] step 3 - exact duplicate column drops (from remediation plan)", file=sys.stderr)
    _emit("step 3 — dropping exact duplicate columns")
    if actions:
        df, drop_risks = _apply_exact_duplicate_column_drops(df, actions)
        unresolved_risks.extend(drop_risks)
    else:
        print("  no remediation plan loaded - skipping exact duplicate column drops.", file=sys.stderr)

    print("\n[apply] step 4 - column renames (from schema cache)", file=sys.stderr)
    _emit("step 4 — applying schema renames")
    df, rename_map = _apply_column_renames(df, path)
    if rename_map:
        _emit(f"  renamed {len(rename_map)} columns")
    for action in actions:
        if action.action_type != "rename_column":
            continue
        column_name = str(action.target.get("column_name", ""))
        new_name = str(action.target.get("new_name", ""))
        if rename_map.get(column_name) == new_name:
            action.status = "applied"
        elif column_name not in df.columns:
            action.status = "not_needed"
        else:
            action.status = "not_needed"

    print("\n[apply] step 5 - dtype casting (from schema cache)", file=sys.stderr)
    _emit("step 5 — casting dtypes")
    df, cast_results = _apply_dtype_casts(df, path)
    _applied_casts = sum(1 for s in cast_results.values() if s == "applied")
    if _applied_casts:
        _emit(f"  {_applied_casts} dtype cast{'s' if _applied_casts != 1 else ''} applied")
    for action in actions:
        if action.action_type != "cast_dtype":
            continue
        column_name = str(action.target.get("column_name", ""))
        status = cast_results.get(column_name, "not_needed")
        action.status = status if status in {"applied", "failed"} else "not_needed"

    print("\n[apply] step 6 - lowercase string columns", file=sys.stderr)
    _emit("step 6 — lowercasing string columns")
    df, lowercased_total, lowercased_by_column = _apply_string_lowercasing(df, path)
    if lowercased_total:
        _emit(
            f"  lowercased {lowercased_total} string value{'s' if lowercased_total != 1 else ''} "
            f"across {len(lowercased_by_column)} column{'s' if len(lowercased_by_column) != 1 else ''}"
        )

    output_dir = cleaning_cache_dir(path)
    output_dir.mkdir(parents=True, exist_ok=True)
    cleaned_path = cleaned_dataset_path(path)
    df.to_csv(cleaned_path, index=False)
    print(f"\n[apply] cleaned dataset saved -> {cleaned_path}", file=sys.stderr)

    save_cleaner_manifest(path, applied_artifacts)
    if remediation_plan is not None:
        save_remediation_plan(path, remediation_plan)

    cleaning_report = CleaningReport(
        dataset_name=path.stem,
        rows_before=rows_before,
        rows_after=len(df),
        columns_before=columns_before,
        columns_after=len(df.columns),
        generated_cleaners=applied_artifacts,
        unresolved_risks=unresolved_risks,
        cleaned_csv_gzip_base64=gzip_text_to_base64(df.to_csv(index=False)),
        summary=(
            f"Applied {len(applied_artifacts)} format cleaners, replaced {total_replaced} placeholder values, "
            f"renamed {len(rename_map)} columns, cast dtypes, and lowercased {lowercased_total} string values. "
            f"Cleaned dataset saved to `{cleaned_path.as_posix()}`."
        ),
    )
    return cleaning_report, execution_reports, remediation_plan


### 6.3.6 `run_verify`

Role: post-cleaning verification stage. It re-runs format consistency on the cleaned file and compares before-vs-after findings to label issues as resolved, improved, unchanged, regressed, or new.


In [ ]:
# run_verify: re-runs format-consistency validation on the cleaned CSV and diffs against the original.
# Each finding becomes one FindingDiff with status resolved / improved / unchanged / regressed.
# Inner helpers:
# - _schema_rename_map                      -> aligns pre/post column names through renames
# - run_format_consistency_validation       -> post-clean consistency pass
# - _compute_diffs / _diff_summary          -> builds the diff report

def run_verify(path: Path, on_event=None, max_workers: int = 1) -> ConsistencyVerificationReport:
    if max_workers < 1:
        raise ValueError("max_workers must be at least 1.")

    def _emit(message: str) -> None:
        if on_event is None:
            return
        try:
            on_event(message)
        except Exception:
            pass

    cleaned_path = cleaned_dataset_path(path)
    if not cleaned_path.exists():
        raise FileNotFoundError(
            f"Cleaned dataset not found at {cleaned_path}. Run --stage apply first."
        )

    original = load_consistency(path)
    original_map = {finding.column_name: finding for finding in original.format_consistency_findings}

    rename_map = _schema_rename_map(path)
    reverse_rename = {new: old for old, new in rename_map.items()}

    cleaned_df = load_dataset_frame(cleaned_path)
    numeric_original_names = _numeric_original_names(cleaned_df, reverse_rename)

    print(f"\n[verify] running consistency on cleaned dataset: {cleaned_path}", file=sys.stderr)
    _emit(f"re-running consistency on {len(original_map)} original finding columns")
    after = run_format_consistency_validation(
        cleaned_path,
        reuse_cache=False,
        read_as_str=True,
        max_workers=max_workers,
    )
    after_map = {
        reverse_rename.get(finding.column_name, finding.column_name): finding
        for finding in after.format_consistency_findings
        if reverse_rename.get(finding.column_name, finding.column_name) not in numeric_original_names
    }

    diffs: list[FindingDiff] = []
    for column_name, before_finding in original_map.items():
        after_finding = after_map.get(column_name)
        before_rows = before_finding.inconsistent_rows
        after_rows = after_finding.inconsistent_rows if after_finding else 0
        reduction_pct = round((before_rows - after_rows) / before_rows * 100, 1) if before_rows > 0 else 0.0

        if after_finding is None or after_rows == 0:
            status = "resolved"
        elif after_rows < before_rows:
            status = "improved"
        elif after_rows == before_rows:
            status = "unchanged"
        else:
            status = "regressed"

        diffs.append(
            FindingDiff(
                column_name=column_name,
                status=status,
                before_inconsistent_rows=before_rows,
                after_inconsistent_rows=after_rows,
                reduction_pct=reduction_pct,
                remaining_examples=after_finding.example_inconsistent_values if after_finding else [],
                renamed_to=rename_map.get(column_name),
            )
        )
        display_name = rename_map.get(column_name, column_name)
        _emit(f"  {status}: '{display_name}' ({before_rows}→{after_rows} rows)")

    for column_name, after_finding in after_map.items():
        if column_name in original_map:
            continue
        diffs.append(
            FindingDiff(
                column_name=column_name,
                status="new",
                before_inconsistent_rows=0,
                after_inconsistent_rows=after_finding.inconsistent_rows,
                reduction_pct=-100.0,
                remaining_examples=after_finding.example_inconsistent_values,
            )
        )

    _print_diff_table(diffs, len(original_map), len(after_map))

    return ConsistencyVerificationReport(
        dataset_name=path.stem,
        original_finding_count=len(original_map),
        remaining_finding_count=len(after_map),
        diffs=diffs,
        summary=_diff_summary(diffs),
    )


### 6.3.7 `build_final_report`

Role: deterministic merger that consolidates validation outputs, remediation actions, cleaning results, and verification diffs into one authoritative report object.


In [ ]:
# build_final_report: deterministic merger. No LLM.
# Groups remediation actions by status, merges cleaning + verification diffs,
# and produces the single authoritative FinalPipelineReport.
# Inner helpers:
# - _group_actions_by_status                -> applied / proposed_not_applied / failed / not_needed

def build_final_report(
    validation_results: OrchestrationStepResult,
    remediation_plan: RemediationPlan,
    cleaning_report: CleaningReport,
    verification_report: ConsistencyVerificationReport | None,
    dataset_path: Path | None = None,
) -> FinalPipelineReport:
    validation_summary = {
        "schema_issues": len(validation_results.schema_validation.issues),
        "completeness_columns_with_missing": len(validation_results.completeness_analysis.columns_with_missing_values),
        "consistency_findings": len(validation_results.consistency_validation.format_consistency_findings),
        "anomaly_findings": len(validation_results.anomaly_detection.findings) if validation_results.anomaly_detection else 0,
        "cross_column_findings": len(validation_results.cross_column_validation.findings) if validation_results.cross_column_validation else 0,
        "duplicate_groups": len(validation_results.duplicate_detection.groups) if validation_results.duplicate_detection else 0,
    }

    applied_actions = [action for action in remediation_plan.actions if action.status == "applied"]
    proposed_not_applied_actions = [action for action in remediation_plan.actions if action.status == "proposed_not_applied"]
    failed_actions = [action for action in remediation_plan.actions if action.status == "failed"]
    not_needed_actions = [action for action in remediation_plan.actions if action.status == "not_needed"]
    duplicate_row_drop_candidates = [
        action for action in remediation_plan.actions if action.action_type == "drop_rows_candidate"
    ]
    manual_review_queue = [
        action
        for action in remediation_plan.actions
        if action.status == "proposed_not_applied"
    ]

    verification_summary = verification_report.summary if verification_report is not None else "Verification was not run."
    summary = (
        f"Validation found {sum(validation_summary.values())} section-level findings/signals. "
        f"Applied {len(applied_actions)} remediation actions, left {len(proposed_not_applied_actions)} proposed without auto-apply, "
        f"and recorded {len(failed_actions)} failed actions."
    )

    total_rows_cleaned, non_null_counts_cleaned = _compute_cleaned_non_null_counts(dataset_path)

    anomaly_findings = (
        list(validation_results.anomaly_detection.findings)
        if validation_results.anomaly_detection is not None
        else []
    )
    cross_column_findings = (
        list(validation_results.cross_column_validation.findings)
        if validation_results.cross_column_validation is not None
        else []
    )
    duplicate_groups = (
        list(validation_results.duplicate_detection.groups)
        if validation_results.duplicate_detection is not None
        else []
    )
    completeness_details = list(validation_results.completeness_analysis.per_column)

    return FinalPipelineReport(
        dataset_name=validation_results.schema_validation.dataset_name,
        validation_summary=validation_summary,
        applied_actions=applied_actions,
        proposed_not_applied_actions=proposed_not_applied_actions,
        failed_actions=failed_actions,
        not_needed_actions=not_needed_actions,
        duplicate_row_drop_candidates=duplicate_row_drop_candidates,
        manual_review_queue=manual_review_queue,
        cleaning_summary=cleaning_report.summary,
        verification_summary=verification_summary,
        verification_diffs=verification_report.diffs if verification_report is not None else [],
        generated_cleaners=cleaning_report.generated_cleaners,
        total_rows_cleaned=total_rows_cleaned,
        non_null_counts_cleaned=non_null_counts_cleaned,
        completeness_details=completeness_details,
        anomaly_findings=anomaly_findings,
        cross_column_findings=cross_column_findings,
        duplicate_groups=duplicate_groups,
        unresolved_risks=cleaning_report.unresolved_risks,
        summary=summary,
    )


### 6.3.8 `generate_narrative_report`

Role: presentation layer for the final report. It turns the deterministic final report into human-readable prose using the chunked narrative agents, with a deterministic fallback if the model output fails.


In [ ]:
# generate_narrative_report: presentation layer. Turns FinalPipelineReport into Markdown prose.
# Uses the chunked narrative agents (frontmatter + one agent per section) so each LLM call is small
# and the final document stays coherent.
# Inner helpers:
# - _build_narrative_briefing               -> compresses the factual report into a compact brief
# - narrative_frontmatter_agent / narrative_section_agent -> per-chunk prose

def generate_narrative_report(final_report: FinalPipelineReport) -> NarrativeReport:
    try:
        return _generate_narrative_report_chunked(final_report)
    except Exception as error:
        print(
            f"[report] warning: chunked narrative generation failed, using deterministic fallback: {error}",
            file=sys.stderr,
        )
        return _fallback_narrative_report(final_report)


## 7. Validation half

Six read-only stages. Each stage function caches its structured output under `Data/.validation_cache/`. The sub-sections below run each stage in turn and display a summary + a findings DataFrame.


### 7.1 Schema validation (dtype inference + naming check)

`run_schema_validation` calls two agents: `dtype_inference_agent` (per-column cleaned dtype + pattern) and `schema_summary_agent` (handoff narrative). It also runs a deterministic naming check (lowercase snake_case) and a duplicate-semantic detector.


In [ ]:
from validation.schema import run_schema_validation  # 2 LLM calls + naming/duplicate heuristics

schema_handoff = run_schema_validation(DATASET_PATH)

display(Markdown(f"**Summary:** {schema_handoff.summary}"))

pd.DataFrame([c.model_dump() for c in schema_handoff.columns])[
    ["name", "pandas_dtype", "numeric_role", "string_role", "detected_pattern", "naming_valid", "rename_suggestion"]
]


### 7.2 Completeness analysis

Profiles non-null counts, missing-like percentages, and placeholder tokens per column, then lets `completeness_analysis_agent` produce a structured report with per-column `recommended_action`.


In [ ]:
from validation.completeness import run_completeness_analysis  # 1 LLM call

completeness = run_completeness_analysis(DATASET_PATH)

display(Markdown(f"**Summary:** {completeness.summary}"))

pd.DataFrame([f.model_dump() for f in completeness.per_column])


### 7.3 Format consistency

For each column we build a `ColumnFormatFacts` profile (dominant value shape, outlier examples). If the schema already carries a `detected_pattern`, we take the **fast path** and build the finding directly from the profile (no LLM). Otherwise we fall through to `format_consistency_agent` for a judgement call.

This stage is what the cleaner pipeline keys off: any finding here becomes a cleaner-generation request in section 8.


In [ ]:
from validation.consistency import run_format_consistency_validation  # fast path + slow path

consistency = run_format_consistency_validation(DATASET_PATH)

display(Markdown(f"**Summary:** {consistency.summary}"))

pd.DataFrame([f.model_dump() for f in consistency.format_consistency_findings])[
    ["column_name", "expected_pattern", "inconsistent_rows", "example_inconsistent_values"]
]


### 7.4 Anomaly detection (numeric outliers + rare categories)

Two heuristic detectors find numeric IQR outliers and rare categorical values. `anomaly_summary_agent` then writes a short downstream summary.


In [ ]:
from validation.anomaly import run_anomaly_detection  # heuristics + 1 summary LLM call
anomaly = run_anomaly_detection(DATASET_PATH)
display(Markdown(f"**Summary:** {anomaly.summary}"))
pd.DataFrame([f.model_dump() for f in anomaly.findings]) if anomaly.findings else "(no anomalies)"


### 7.5 Cross-column validation

Heuristic checks: near-duplicate columns (by value similarity), duplicate-semantic conflicts (columns that should agree but don't), year/month period mismatches, and date-order violations. `cross_column_summary_agent` narrates.


In [ ]:
from validation.cross_column import run_cross_column_validation  # heuristics + 1 summary LLM call
cross_column = run_cross_column_validation(DATASET_PATH)
display(Markdown(f"**Summary:** {cross_column.summary}"))
pd.DataFrame([f.model_dump() for f in cross_column.findings]) if cross_column.findings else "(no cross-column findings)"


### 7.6 Duplicate record detection

Exact + near-duplicate row groups, plus key-column inference. `duplicate_summary_agent` narrates volumes but never recommends deletion — that is a human decision.


In [ ]:
from validation.duplicates import run_duplicate_detection  # heuristics + 1 summary LLM call
duplicates = run_duplicate_detection(DATASET_PATH)
display(Markdown(f"**Summary:** {duplicates.summary}"))
pd.DataFrame([g.model_dump() for g in duplicates.groups]) if duplicates.groups else "(no duplicate groups)"


### 7.7 Bundling the validation results

`build_validation_results` is the one-shot orchestrator used by the CLI and by the cleaning pipeline. Here we call it with the cached artifacts from the stages above — it loads them instead of re-running the LLM calls.


In [ ]:
from validation import build_validation_results  # runs (or reuses) all six validation stages
validation_results = build_validation_results(
    DATASET_PATH,
    reuse_schema=True,
    reuse_completeness=True,
    reuse_consistency=True,
)
print("OrchestrationStepResult fields:", list(validation_results.model_dump().keys()))


## 8. Remediation plan

`run_remediation_planning` is **deterministic** — no LLM call. It walks the validation bundle and produces a flat list of `RemediationAction`s. Each action carries an `action_type`, a target, a `confidence` / `risk_level`, and a crucial `auto_apply` boolean distinguishing safe actions the pipeline will perform automatically from proposals that need human review.

Action types:

* `rename_column`, `cast_dtype`, `replace_placeholders_with_null`, `generate_cleaner`, `drop_exact_duplicate_column` — typically `auto_apply=True`.
* `manual_review`, `report_only`, `drop_rows_candidate` — never auto-applied; reported only.


In [ ]:
from cleaning.remediation import run_remediation_planning  # deterministic, caches a JSON plan
remediation_plan = run_remediation_planning(DATASET_PATH, validation_results=validation_results)
print(remediation_plan.summary)
pd.DataFrame([a.model_dump() for a in remediation_plan.actions])[
    ["action_id", "action_type", "object_type", "auto_apply", "risk_level", "status", "reason"]
]


## 9. Cleaning half — the generator / critic loop

The cleaning half is where per-column Python cleaning functions are synthesised. For each format-consistency finding we build a `ColumnCleaningRequest` — a self-contained bundle of schema, completeness, and format evidence — and run a retry loop:

1. **Prompt** the generator with the request (plus any previous failure context).
2. The **generator agent** writes a Python cleaner and runs **exactly one** grouped code-execution check.
3. The **host-side validator** (no LLM) checks the program against every dominant value (must be unchanged) and every outlier (must be transformed or nulled), plus structural rules (target dtype, pattern, no outer-scope dependencies, no shadowed delimiter branches).
4. If issues remain, the **critic agent** diagnoses the smallest credible repair and the diagnosis is fed forward to the next attempt.
5. A **stagnation detector** (same code or same issue fingerprint twice in a row) bumps the generator temperature `0.2 ? 0.3 ? 0.4 ? 0.5` and injects a structural rewrite skeleton into the prompt.

The per-attempt LLM call is capped at **one** grouped code-execution tool call (`UsageLimits(tool_calls_limit=1)`) — this is the key architectural choice that prevents hidden self-repair loops inside a single model run and forces host-side code to own correctness.


### 9.1 Building one `ColumnCleaningRequest`

`build_column_cleaning_request` merges three pieces of evidence into the generator's input: the schema entry (dtype, pattern), the format profile (dominant shape + full list of outlier examples), and the consistency finding (expected pattern + `suggested_strategy`). The `suggested_strategy` is the authoritative contract the generator must implement.


In [ ]:
from cleaning.request import build_column_cleaning_request  # merges schema + format facts into a request
from tools import build_column_format_facts                 # dominant shape + outlier examples per column

schema_map = {c.name: c for c in schema_handoff.columns}
example_finding = consistency.format_consistency_findings[0]
example_facts = build_column_format_facts(raw_df, example_finding.column_name)
example_request = build_column_cleaning_request(
    DATASET_PATH.stem,
    example_finding.column_name,
    example_finding,
    example_facts,
    schema_map.get(example_finding.column_name),
)
print(example_request.model_dump_json(indent=2))


### 9.2 Host-side validator — what can go wrong

The validator emits one of these `CleanerValidationIssue` categories. None require an LLM — they are pure Python checks against the generated code.

| Category | Meaning |
|---|---|
| `program_mismatch` | Program declares a different column name than the request. |
| `non_self_contained_function` | Cleaner references outer-scope names (NameError on load or call). |
| `runtime_exception` | Cleaner raised any other exception. |
| `shadowed_specific_branch` | Generic `if '<sep>' in s:` above a more specific branch on the same separator. |
| `dominant_value_modified` | Cleaner rewrote an already-valid dominant example — identity violation. |
| `outlier_unchanged` | Cleaner returned an inconsistent example unchanged. |
| `wrong_output_shape` | Output value shape does not match the dominant output shape. |
| `not_parseable_as_target_dtype` | Cleaned value does not parse as `Int64` / `Float64` / `datetime64[ns]` / `boolean`. |
| `not_matching_target_pattern` | Cleaned value does not match the numeric schema pattern (e.g. `YYYYMM`). |


### 9.3 The per-column loop

Below is the **real, executing source** of `run_column_cleaner_program` — displayed via `inspect.getsource`. Progress reporting (stderr prints + optional UI callback) is encapsulated in a `_GenerationProgress` object so the control flow reads cleanly.


In [ ]:
from cleaning.generation import run_column_cleaner_program, GENERATOR_USAGE_LIMITS, _stagnation_temperature
print(f"GENERATOR_USAGE_LIMITS = {GENERATOR_USAGE_LIMITS}")
print(f"stagnation ramp 0..5: {[round(_stagnation_temperature(n), 2) for n in range(1, 6)]}")
print()
print(inspect.getsource(run_column_cleaner_program))

### 9.4 Generate one cleaner per inconsistent column

`run_cleaner_generation` drives the loop over every format-consistency finding, saves each accepted program to `Data/.cleaning_cache/<dataset>/generated_cleaners/<column>.py`, and writes `cleaner_manifest.json`.


In [ ]:
from cleaning.generation import run_cleaner_generation  # driver: generator/critic loop per column
artifacts = run_cleaner_generation(DATASET_PATH, reuse_consistency=True, max_attempts=10)
for a in artifacts:
    print(f"- {a.column_name}  ->  {a.code_path}")
    print(f"    {a.summary}")

### 9.5 Inspect one generated cleaner (from disk)

The generator's output is written to disk as a regular `.py` file. That is the cleaner that will be executed in the apply stage.


In [ ]:
if artifacts:
    cleaner_src = Path(artifacts[0].code_path).read_text(encoding="utf-8")
    print(f"# {artifacts[0].code_path}\n")
    print(cleaner_src)

### 9.6 Apply the remediation plan + generated cleaners

Actions are applied in a fixed, safe order:

1. Column renames (from schema suggestions)
2. Placeholder → null replacements (from completeness findings)
3. Generated per-column cleaners (from this section)
4. Exact duplicate column drops
5. Explicit dtype casts

The cleaned CSV is written to `Data/.cleaning_cache/<dataset>/<dataset>.cleaned.csv`.


In [ ]:
from cleaning.application import run_cleaner_application_with_plan  # applies renames + nulls + cleaners + casts
cleaning_report, execution_reports, applied_plan = run_cleaner_application_with_plan(DATASET_PATH, remediation_plan)
print(cleaning_report.summary)
if execution_reports:
    display(pd.DataFrame([r.model_dump(exclude={"sample_updates"}) for r in execution_reports]))


### 9.7 Verification — before vs. after

`run_verify` re-runs the format-consistency validation on the cleaned CSV (read as raw strings so pandas cannot silently re-normalise formats) and produces a per-column `FindingDiff` with status `resolved` / `improved` / `unchanged` / `regressed` / `new`.


In [ ]:
from cleaning.verification import run_verify  # re-runs consistency on the cleaned CSV
verification_report = run_verify(DATASET_PATH)
print(verification_report.summary)
pd.DataFrame([d.model_dump() for d in verification_report.diffs])


## 10. Final report + narrative

The pipeline's closing act: merge every artifact into a `FinalPipelineReport`, write it to `<dataset>.final_report.json`, then invoke `generate_narrative_report(...)`, which now uses `narrative_frontmatter_agent` plus `narrative_section_agent` (with deterministic fallback) to produce the final Markdown report for human review.


In [ ]:
from cleaning.reporting import (
    build_final_report,            # pure model merge, no LLM
    save_final_report,             # writes final_report.json
    generate_narrative_report,     # calls narrative_report_agent
    save_narrative_report,         # writes narrative_report.md
)

final_report = build_final_report(
    validation_results,
    applied_plan,
    cleaning_report,
    verification_report,
    dataset_path=DATASET_PATH,
)
save_final_report(DATASET_PATH, final_report)
narrative = generate_narrative_report(final_report)
narrative_path = save_narrative_report(DATASET_PATH, narrative)
display(Markdown(narrative_path.read_text(encoding="utf-8")))

## 11. Closing

Everything produced by this run lives under the dataset's parent directory:

```
Data/.validation_cache/spesa.*.json         # schema, completeness, consistency, anomaly, cross_column, duplicates, remediation_plan, validation_bundle
Data/.cleaning_cache/spesa/
    generated_cleaners/*.py                  # one Python cleaner per inconsistent column
    cleaner_manifest.json                    # list of GeneratedCleanerArtifact
    spesa.cleaned.csv                        # fully cleaned dataset
    spesa.final_report.json                  # FinalPipelineReport serialised
    spesa.narrative_report.md                # the document rendered above
```

Re-running the notebook reuses every cached validation artifact (instant) except the cleaning half, which regenerates cleaners each time since their path in the notebook passes no reuse flag. For deeper internals, see `docs/AGENT_ARCHITECTURE.md` and `docs/REMEDIATION_WORKFLOW.md`.


## 12. Artifacts produced on disk

Everything below lives under the dataset's parent directory; the professor can re-examine any step without re-running the LLM calls.

| Path (relative to `Data/`) | Stage | Format |
|---|---|---|
| `.validation_cache/<dataset>.schema_handoff.json` | Schema validation | JSON |
| `.validation_cache/<dataset>.completeness.json` | Completeness | JSON |
| `.validation_cache/<dataset>.consistency.json` | Format consistency | JSON |
| `.validation_cache/<dataset>.anomaly.json` | Anomaly detection | JSON |
| `.validation_cache/<dataset>.cross_column.json` | Cross-column | JSON |
| `.validation_cache/<dataset>.duplicates.json` | Duplicate detection | JSON |
| `.validation_cache/<dataset>.validation_bundle.json` | Bundled validation | JSON |
| `.validation_cache/<dataset>.remediation_plan.json` | Remediation plan | JSON |
| `.cleaning_cache/<dataset>/generated_cleaners/*.py` | One cleaner per dirty column | Python |
| `.cleaning_cache/<dataset>/cleaner_manifest.json` | Accepted cleaners index | JSON |
| `.cleaning_cache/<dataset>/<dataset>.cleaned.csv` | Cleaned output | CSV |
| `.cleaning_cache/<dataset>/<dataset>.final_report.json` | Final structured report | JSON |
| `.cleaning_cache/<dataset>/<dataset>.narrative_report.md` | Final human-readable report | Markdown |
